<a href="https://colab.research.google.com/github/rahavi-r31/ExporterAI_Chapter_68_analytics/blob/colab/product_description_cleaning_jan_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================
HSN CHAPTER 52 - COTTON - COMPREHENSIVE ANALYZER V2
================================================================================
FLOW:
1. Upload DATA FILE (with PRODUCT DESCRIPTION + HS CODE columns)
2. Upload HSN FILE (with HSN_CD + HSN_Description columns)
3. Code merges, cleans, processes ALL through LLM (15 per batch)
4. Outputs grouped tags (Usage, Processing, Material, Weave, Other)

For Google Colab - Run each cell in order
================================================================================
"""

# ============================================================================
# CELL 1: INSTALL & IMPORT
# ============================================================================

!pip install -q pandas openpyxl openai

import pandas as pd
import numpy as np
import re
import time
import json
from collections import Counter, defaultdict
from datetime import datetime
from openai import OpenAI
from google.colab import files

print("✅ All dependencies installed!")
print(f"📅 Session started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# CELL 2: UPLOAD FILES
# ============================================================================

print("="*60)
print("📂 STEP 1: Upload your DATA FILE")
print("   (Must have columns: PRODUCT DESCRIPTION, HS CODE)")
print("="*60)
uploaded_data = files.upload()
DATA_FILE = '/content/' + list(uploaded_data.keys())[0]
print(f"✅ Data file uploaded: {DATA_FILE}")

print("\n" + "="*60)
print("📂 STEP 2: Upload your HSN MAPPING FILE")
print("   (Must have columns: HSN_CD, HSN_Description)")
print("="*60)
uploaded_hsn = files.upload()
HSN_FILE = '/content/' + list(uploaded_hsn.keys())[0]
print(f"✅ HSN file uploaded: {HSN_FILE}")

# ============================================================================
# CELL 3: LOAD & MERGE FILES
# ============================================================================

print("\n" + "="*60)
print("📊 Loading and merging files...")
print("="*60)

# Load data file
if DATA_FILE.endswith('.xlsx') or DATA_FILE.endswith('.xls'):
    data = pd.read_excel(DATA_FILE)
else:
    data = pd.read_csv(DATA_FILE, encoding='utf-8', on_bad_lines='skip')

print(f"✅ Data file: {len(data):,} records")
print(f"   Columns: {list(data.columns)}")

# Load HSN mapping file
if HSN_FILE.endswith('.xlsx') or HSN_FILE.endswith('.xls'):
    hsn_map = pd.read_excel(HSN_FILE)
else:
    hsn_map = pd.read_csv(HSN_FILE, encoding='utf-8', on_bad_lines='skip')

print(f"✅ HSN file: {len(hsn_map):,} mappings")
print(f"   Columns: {list(hsn_map.columns)}")

# Standardize HS CODE columns
data['HS_CODE'] = data['HS CODE'].astype(str).str.strip().str.zfill(8)
hsn_map['HSN_CD'] = hsn_map['HSN_CD'].astype(str).str.strip().str.zfill(8)

# Merge HSN descriptions into data
data = data.merge(
    hsn_map[['HSN_CD', 'HSN_Description']],
    left_on='HS_CODE',
    right_on='HSN_CD',
    how='left'
)

# Add hierarchy columns
data['Chapter'] = data['HS_CODE'].str[:2]
data['Heading'] = data['HS_CODE'].str[:4]
data['Sub_Heading'] = data['HS_CODE'].str[:6]

print(f"\n✅ Merged! Records with HSN description: {data['HSN_Description'].notna().sum():,}")
print(f"   Records without HSN description: {data['HSN_Description'].isna().sum():,}")

# ============================================================================
# CELL 4: CONFIGURATION
# ============================================================================

# LLM Configuration
LLM_BASE_URL = "https://api.llm7.io/v1"
LLM_API_KEY = "unused"
LLM_MODEL = "meta-llama/Llama-3.1-8B-Instruct"
BATCH_SIZE = 15
MIN_DELAY_SECONDS = 5
MAX_RETRIES = 3

# Tag Groups
TAG_GROUPS = {
    "Usage": [
        "Shirting", "Suiting", "Blazer Cloth", "Blouse Cloth", "Uniform/Workwear",
        "Home Textiles", "Ladies Garments", "Kids/Baby", "Medical/Healthcare",
        "Shirts", "Innerwear", "Suits", "Unstitched Material", "Dress Material",
        "Blazers / Jackets", "Trousers / Pants", "Sarees / Ethnic Wear",
        "Home Furnishing", "Readymade Garments", "Shawls / Religious",
        "Lungi (Traditional Garment)", "Dhoti (Traditional Garment)", "Gamcha (Towel/Workwear)",
        "Jhola (Bag)", "Bedding (Razai/Quilt Cover)", "Religious/Ceremonial Cloth"
    ],
    "Processing": [
        "Grey/Greige", "Yarn Dyed", "Indigo Yarn Dyed", "Printed", "Digital Printed",
        "Dyed", "Bleached/White", "Embroidered", "Mercerised", "Finished",
        "Washed", "Singed", "Handloom", "Powerloom"
    ],
    "Material": [
        "100% Cotton", "Cotton Blends", "Cotton Fabric", "Polyester Fabric", "Fabric / Textile",
        "Premium/Luxury", "Synthetic", "Eco-Friendly", "Organic", "Combed",
        "Pure Linen", "Linen-Cotton Blend", "Rayon/Viscose", "Rayon-Cotton Blend",
        "Wool", "Denim"
    ],
    "Weave/Construction": [
        "Poplin", "Mattress Cloth", "Fine Cotton Weaves", "Structured Weaves", "Flannel",
        "Traditional Indian", "Chikan Embroidery", "Malmal (Muslin)", "Markin Cotton",
        "Rubiya Cotton", "Dashna Cotton", "Patra Cotton", "Chalti/Chalte Cotton",
        "Tanna Cotton", "Traditional Cotton Varieties", "Knit/Jersey", "Heavy Duty",
        "Net/Mesh", "Checked Fabric", "Thread / Yarn"
    ],
    "Other": [
        "SAMPLE", "Garment Accessories", "Unclassified", "Unclassified (Brand/Invoice Reference)",
        "MEIS", "FIEMA", "Mixed / Assorted", "Documentation / Compliance"
    ]
}

# Error Codes
ERROR_CODES = {
    "E001": "Connection timeout",
    "E002": "Rate limit exceeded",
    "E003": "Invalid API response",
    "E004": "JSON parsing failed",
    "E005": "Empty response from LLM",
    "E006": "Model overloaded",
    "E007": "Authentication failed",
    "E008": "Unknown error"
}

print("✅ Configuration loaded!")

# ============================================================================
# CELL 5: CLEANING FUNCTIONS
# ============================================================================

class CleaningStats:
    def __init__(self):
        self.total_records = 0
        self.noise_removed = defaultdict(int)
        self.patterns_matched = defaultdict(int)
        self.chars_removed = 0
        self.chars_original = 0
        self.empty_after_clean = 0

    def report(self):
        print("\n" + "="*70)
        print("📊 CLEANING STATISTICS")
        print("="*70)
        print(f"Total records: {self.total_records:,}")
        print(f"Original characters: {self.chars_original:,}")
        print(f"Characters removed: {self.chars_removed:,} ({(self.chars_removed/max(1,self.chars_original))*100:.1f}%)")
        print(f"Empty after cleaning: {self.empty_after_clean}")

        if self.noise_removed:
            print(f"\n🚫 Top 15 Noise Patterns Removed:")
            for pattern, count in sorted(self.noise_removed.items(), key=lambda x: x[1], reverse=True)[:15]:
                print(f"   {pattern:<40}: {count:,}")

        if self.patterns_matched:
            print(f"\n✅ Top 15 Useful Patterns Found:")
            for pattern, count in sorted(self.patterns_matched.items(), key=lambda x: x[1], reverse=True)[:15]:
                print(f"   {pattern:<40}: {count:,}")

# Noise patterns
NOISE_PATTERNS = [
    (r'\bAS\s*PER\s*INVOICE\b', 'AS PER INVOICE'),
    (r'\bAS\s*PER\s*PROFORMA\b', 'AS PER PROFORMA'),
    (r'\bAS\s*PER\s*LC\b', 'AS PER LC'),
    (r'\bAS\s*PER\s*CONTRACT\b', 'AS PER CONTRACT'),
    (r'\bMADE\s*IN\s*INDIA\b', 'MADE IN INDIA'),
    (r'\bMADE\s*IN\s*[A-Z]+\b', 'MADE IN [COUNTRY]'),
    (r'\bFOB\b', 'FOB'),
    (r'\bCIF\b', 'CIF'),
    (r'\bFREE\s*SAMPLE\b', 'FREE SAMPLE'),
    (r'\bNO\s*COMMERCIAL\s*VALUE\b', 'NO COMMERCIAL VALUE'),
    (r'\bPO[:\s#]*[A-Z0-9\-]+\b', 'PO NUMBER'),
    (r'\bINVOICE[:\s#]*[A-Z0-9\-]+\b', 'INVOICE NUMBER'),
    (r'\bSTYLE[:\s#]*[A-Z0-9\-]+\b', 'STYLE NUMBER'),
    (r'\bHS\s*CODE[:\s]*\d+\b', 'HS CODE REF'),
    (r'\bHSN[:\s]*\d+\b', 'HSN REF'),
    (r'\b\d{1,2}[\-/\.]\d{1,2}[\-/\.]\d{2,4}\b', 'DATE'),
    (r'\b[A-Z]{2,4}\d{6,}\b', 'ALPHANUMERIC CODE'),
]

# Useful patterns
USEFUL_PATTERNS = [
    (r'\b100\s*%?\s*COTTON\b', '100% Cotton'),
    (r'\b(\d{2,3})\s*(?:GSM|G/M)\b', 'GSM'),
    (r'\b(\d{2,3})\s*(?:INCH|IN|")\b', 'Width'),
    (r'\b(\d{2,3})\s*[xX]\s*(\d{2,3})\b', 'Thread Count'),
    (r'\bGREY\b', 'Grey'),
    (r'\bBLEACHED\b', 'Bleached'),
    (r'\bDYED\b', 'Dyed'),
    (r'\bPRINTED\b', 'Printed'),
    (r'\bYARN\s*DYED\b', 'Yarn Dyed'),
    (r'\bSHIRTING\b', 'Shirting'),
    (r'\bSUITING\b', 'Suiting'),
    (r'\bSAREE\b', 'Saree'),
    (r'\bDHOTI\b', 'Dhoti'),
    (r'\bLUNGI\b', 'Lungi'),
]

cleaning_stats = CleaningStats()

def clean_description(text):
    global cleaning_stats
    if pd.isna(text) or not str(text).strip():
        cleaning_stats.empty_after_clean += 1
        return ""

    original = str(text)
    cleaning_stats.chars_original += len(original)

    cleaned = original.upper()

    # Track useful patterns
    for pattern, name in USEFUL_PATTERNS:
        if re.search(pattern, cleaned, re.IGNORECASE):
            cleaning_stats.patterns_matched[name] += 1

    # Remove noise
    for pattern, name in NOISE_PATTERNS:
        if re.search(pattern, cleaned, re.IGNORECASE):
            cleaning_stats.noise_removed[name] += 1
            cleaned = re.sub(pattern, ' ', cleaned, flags=re.IGNORECASE)

    # Clean special chars and whitespace
    cleaned = re.sub(r'[^\w\s\.\,\-\/\%\(\)]', ' ', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    cleaning_stats.chars_removed += (len(original) - len(cleaned))
    if not cleaned:
        cleaning_stats.empty_after_clean += 1

    return cleaned

print("✅ Cleaning functions defined!")

# ============================================================================
# CELL 6: APPLY CLEANING
# ============================================================================

print("\n" + "="*60)
print("🧹 Cleaning product descriptions...")
print("="*60)

cleaning_stats = CleaningStats()
cleaning_stats.total_records = len(data)

data['CLEANED_DESCRIPTION'] = data['PRODUCT DESCRIPTION'].apply(clean_description)

cleaning_stats.report()

# ============================================================================
# CELL 7: LLM PROCESSING FUNCTIONS
# ============================================================================

client = OpenAI(base_url=LLM_BASE_URL, api_key=LLM_API_KEY)

def get_error_code(e):
    err = str(e).lower()
    if 'timeout' in err: return 'E001'
    if 'rate limit' in err or '429' in err: return 'E002'
    if 'invalid' in err: return 'E003'
    if 'json' in err: return 'E004'
    if 'empty' in err: return 'E005'
    if 'overload' in err or '503' in err: return 'E006'
    if 'auth' in err or '401' in err: return 'E007'
    return 'E008'

def process_batch(descriptions, indices):
    prompt = f"""Analyze these {len(descriptions)} textile product descriptions. Return ONLY a JSON array.

For each, extract:
- "index": the index number
- "refined_description": clean product name (max 50 words)
- "product_type": category (Shirting/Saree/Dhoti/Lungi/Voile/etc)
- "material": composition (100% Cotton/Cotton Blend/etc)
- "processing": treatment (Grey/Bleached/Dyed/Printed/Yarn Dyed)
- "weave_type": weave (Plain/Twill/Dobby/Jacquard)
- "gsm": weight number or null
- "width": width string or null
- "tags": array of relevant tags

Descriptions:
"""
    for i, desc in zip(indices, descriptions):
        prompt += f"\n[{i}]: {desc}"

    prompt += "\n\nReturn JSON array only:"

    for attempt in range(MAX_RETRIES):
        try:
            print(f"   Attempt {attempt+1}/{MAX_RETRIES}...", end=" ")
            start = time.time()

            response = client.chat.completions.create(
                model=LLM_MODEL,
                messages=[
                    {"role": "system", "content": "Textile expert. Respond with valid JSON only."},
                    {"role": "user", "content": prompt}
                ],
                temperature=0.3,
                max_tokens=3000
            )

            elapsed = time.time() - start
            content = response.choices[0].message.content.strip()

            # Clean markdown
            if '```' in content:
                content = re.sub(r'^```(?:json)?\s*', '', content)
                content = re.sub(r'\s*```$', '', content)

            results = json.loads(content)
            print(f"✅ ({elapsed:.1f}s, {len(results)} items)")

            # Enforce 5-second delay
            if elapsed < MIN_DELAY_SECONDS:
                time.sleep(MIN_DELAY_SECONDS - elapsed)

            return results, None

        except Exception as e:
            code = get_error_code(e)
            print(f"❌ {code}: {ERROR_CODES[code]}")
            if attempt < MAX_RETRIES - 1:
                time.sleep(MIN_DELAY_SECONDS * (attempt + 2))

    # All failed
    return [{"index": i, "refined_description": "ERROR", "product_type": "ERROR",
             "error_code": code, "error_message": ERROR_CODES[code]} for i in indices], code

print("✅ LLM functions defined!")

# ============================================================================
# CELL 8: PROCESS ALL DESCRIPTIONS
# ============================================================================

print("\n" + "="*60)
print("🤖 Processing ALL descriptions through LLM...")
print("="*60)

descriptions = data['CLEANED_DESCRIPTION'].fillna('').tolist()
total = len(descriptions)
batches = (total + BATCH_SIZE - 1) // BATCH_SIZE

print(f"   Total records: {total:,}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Total batches: {batches}")
print(f"   Est. time: {batches * MIN_DELAY_SECONDS / 60:.1f} min")
print("="*60)

all_results = [None] * total
errors = 0
consecutive_rate_limits = 0
MAX_CONSECUTIVE_RATE_LIMITS = 5
CHECKPOINT_EVERY = 10  # Save every 10 batches

def save_checkpoint(data_df, results_list, batch_num):
    """Save progress checkpoint"""
    data_df['llm_results_temp'] = results_list
    checkpoint_file = f'/content/checkpoint_batch_{batch_num}.xlsx'
    data_df.to_excel(checkpoint_file, index=False)
    print(f"   💾 Checkpoint saved: {checkpoint_file}")
    return checkpoint_file

last_checkpoint = None

for b in range(batches):
    start_idx = b * BATCH_SIZE
    end_idx = min(start_idx + BATCH_SIZE, total)

    batch_desc = descriptions[start_idx:end_idx]
    batch_idx = list(range(len(batch_desc)))

    print(f"\n📦 Batch {b+1}/{batches} (rows {start_idx+1}-{end_idx})")

    results, err = process_batch(batch_desc, batch_idx)

    # Track consecutive rate limits
    if err == 'E002':
        consecutive_rate_limits += 1
        print(f"   ⚠️ Rate limit hit {consecutive_rate_limits}/{MAX_CONSECUTIVE_RATE_LIMITS}")
    else:
        consecutive_rate_limits = 0

    for r in results:
        local_i = r.get('index', 0)
        global_i = start_idx + local_i
        if global_i < total:
            all_results[global_i] = r
            if r.get('error_code'):
                errors += 1

    print(f"   Progress: {min(end_idx, total):,}/{total:,} ({100*end_idx/total:.1f}%)")

    # Save checkpoint every N batches
    if (b + 1) % CHECKPOINT_EVERY == 0:
        last_checkpoint = save_checkpoint(data, all_results, b + 1)

    # STOP if too many consecutive rate limits
    if consecutive_rate_limits >= MAX_CONSECUTIVE_RATE_LIMITS:
        print(f"\n🛑 STOPPING: {MAX_CONSECUTIVE_RATE_LIMITS} consecutive rate limit errors!")
        print(f"   Processed {end_idx}/{total} records ({100*end_idx/total:.1f}%)")
        print(f"   Saving what we have...")
        last_checkpoint = save_checkpoint(data, all_results, b + 1)
        break

print(f"\n✅ LLM Processing {'Stopped Early' if consecutive_rate_limits >= MAX_CONSECUTIVE_RATE_LIMITS else 'Complete'}!")
print(f"   Processed: {sum(1 for r in all_results if r is not None):,}/{total:,}")
print(f"   Errors: {errors}")

# ============================================================================
# CELL 9: EXTRACT RESULTS & GROUP TAGS
# ============================================================================

print("\n" + "="*60)
print("🏷️ Extracting results and grouping tags...")
print("="*60)

data['llm_results'] = all_results

# Extract columns
data['Refined_Description'] = data['llm_results'].apply(lambda x: x.get('refined_description') if isinstance(x, dict) else None)
data['Product_Type'] = data['llm_results'].apply(lambda x: x.get('product_type') if isinstance(x, dict) else None)
data['Material'] = data['llm_results'].apply(lambda x: x.get('material') if isinstance(x, dict) else None)
data['Processing'] = data['llm_results'].apply(lambda x: x.get('processing') if isinstance(x, dict) else None)
data['Weave_Type'] = data['llm_results'].apply(lambda x: x.get('weave_type') if isinstance(x, dict) else None)
data['GSM'] = data['llm_results'].apply(lambda x: x.get('gsm') if isinstance(x, dict) else None)
data['Width'] = data['llm_results'].apply(lambda x: x.get('width') if isinstance(x, dict) else None)
data['Error_Code'] = data['llm_results'].apply(lambda x: x.get('error_code') if isinstance(x, dict) else None)
data['Error_Message'] = data['llm_results'].apply(lambda x: x.get('error_message') if isinstance(x, dict) else None)

# Tag grouping
def split_tags(tags_list):
    if not tags_list or not isinstance(tags_list, list):
        return {g: [] for g in TAG_GROUPS}

    result = {g: [] for g in TAG_GROUPS}
    for tag in tags_list:
        found = False
        for group, group_tags in TAG_GROUPS.items():
            if any(gt.lower() in tag.lower() or tag.lower() in gt.lower() for gt in group_tags):
                result[group].append(tag)
                found = True
                break
        if not found:
            result["Other"].append(tag)
    return result

grouped = data['llm_results'].apply(lambda x: split_tags(x.get('tags', [])) if isinstance(x, dict) else {g: [] for g in TAG_GROUPS})

for group in TAG_GROUPS:
    data[f'Tags_{group}'] = grouped.apply(lambda x: " | ".join(x[group]) if x[group] else None)

data['SUB_CATEGORY'] = data['llm_results'].apply(lambda x: " | ".join(x.get('tags', [])) if isinstance(x, dict) else None)

print("✅ Tag grouping complete!")
print(f"   Created: Tags_Usage, Tags_Processing, Tags_Material, Tags_Weave/Construction, Tags_Other")

# ============================================================================
# CELL 10: SAVE & DOWNLOAD
# ============================================================================

print("\n" + "="*60)
print("💾 Saving results...")
print("="*60)

# Drop dict column
output = data.drop(columns=['llm_results'], errors='ignore')

# Save
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'/content/chapter52_analyzed_{timestamp}.xlsx'
output.to_excel(output_file, index=False)
print(f"✅ Saved: {output_file}")

# Download
print("\n📥 Downloading file...")
files.download(output_file)

print("\n" + "="*60)
print("🎉 COMPLETE!")
print("="*60)

In [ ]:
"""
================================================================================
HSN CHAPTER 52 - COTTON - ANALYZER V4 (RULE-BASED HEAVY)
================================================================================
- 80% Rule-based extraction (keywords, regex, patterns)
- 20% AI (only for formatting messy descriptions - OPTIONAL)
- Comprehensive Indian textile industry keywords
- Fast processing - minimal API calls

For Google Colab - Run each cell in order
================================================================================
"""

# CELL 1: INSTALL & IMPORT
!pip install -q pandas openpyxl openai

import pandas as pd
import numpy as np
import re
import time
import json
from collections import defaultdict
from datetime import datetime
from google.colab import files

print("✅ Dependencies installed!")

# CELL 2: UPLOAD FILES
print("="*60)
print("📂 Upload DATA FILE (HS CODE, PRODUCT DESCRIPTION)")
print("="*60)
uploaded_data = files.upload()
DATA_FILE = '/content/' + list(uploaded_data.keys())[0]

print("\n📂 Upload HSN MAPPING FILE (HSN_CD, HSN_Description)")
uploaded_hsn = files.upload()
HSN_FILE = '/content/' + list(uploaded_hsn.keys())[0]

# CELL 3: KEYWORD DICTIONARIES
PRODUCT_KEYWORDS = {
    # RAW MATERIALS
    'Raw Cotton': ['RAW COTTON', 'COTTON BALE', 'LINT', 'NOT CARDED', 'NOT COMBED', 'KAPAS'],
    'Comber Noil': ['COMBER NOIL', 'COMBER', 'NOIL', 'COMBING WASTE'],
    'Cotton Waste': ['COTTON WASTE', 'WASTE', 'DROPPINGS', 'SWEEPINGS', 'HARD WASTE'],
    'Flat Strip': ['FLAT STRIP', 'FLAT WASTE', 'CARD WASTE'],
    'Cotton Sliver': ['SLIVER', 'COTTON SLIVER', 'DRAWING SLIVER'],
    'Cotton Roving': ['ROVING', 'COTTON ROVING', 'SPEED FRAME'],
    'Absorbent Cotton': ['ABSORBENT', 'SURGICAL COTTON', 'MEDICAL COTTON'],
    # THREAD
    'Sewing Thread': ['SEWING THREAD', 'STITCHING THREAD'],
    'Embroidery Thread': ['EMBROIDERY', 'EMBROID', 'ZARI THREAD'],
    'Pooja Thread': ['POOJA', 'PUJA', 'KALAWA', 'MAULI', 'MOLI', 'JANEU', 'SACRED THREAD', 'DHAGA'],
    'Cotton Wick': ['WICK', 'BATTI', 'DIYA BATTI', 'LAMP WICK'],
    # YARN
    'Combed Yarn': ['COMBED YARN', 'COMBED', 'CBD'],
    'Carded Yarn': ['CARDED YARN', 'CARDED', 'CRD'],
    'Compact Yarn': ['COMPACT', 'COMPACT YARN', 'ELI TWIST'],
    'Ring Spun Yarn': ['RING SPUN', 'RING YARN', 'RS'],
    'Open End Yarn': ['OPEN END', 'OE YARN', 'ROTOR YARN', 'ROTOR SPUN'],
    'Vortex Yarn': ['VORTEX', 'MVS', 'MURATA'],
    'Core Spun Yarn': ['CORE SPUN', 'CORE YARN', 'CSY', 'LYCRA CORE'],
    'Melange Yarn': ['MELANGE', 'MELANG', 'MARL', 'HEATHER'],
    'Slub Yarn': ['SLUB', 'SLUBBED', 'NEPPY'],
    'Knitting Yarn': ['KNITTING', 'KNIT', 'HOSIERY', 'HOS', 'FOR KNITTING'],
    'Weaving Yarn': ['WEAVING', 'WEAV', 'WARP', 'WEFT', 'FOR WEAVING'],
    # TRADITIONAL INDIAN
    'Saree': ['SAREE', 'SARI', 'TANT', 'TAANT', 'JAMDANI', 'KANJIVARAM', 'BANARASI', 'CHANDERI',
              'MAHESHWARI', 'SAMBALPURI', 'IKAT', 'PATOLA', 'POCHAMPALLY'],
    'Dhoti': ['DHOTI', 'DHOTIES', 'PANCHA', 'VESHTI', 'MUNDU'],
    'Lungi': ['LUNGI', 'LUNGIES', 'KAILI'],
    'Gamcha': ['GAMCHA', 'GAMCHHA', 'GAMUCHA', 'ANGOCHHA'],
    'Dupatta': ['DUPATTA', 'CHUNNI', 'ODHNI', 'STOLE'],
    # APPAREL FABRICS
    'Shirting': ['SHIRTING', 'SHIRT FABRIC', 'SHIRT FAB', 'SHIRTINGS'],
    'Suiting': ['SUITING', 'SUIT FABRIC', 'TROUSER', 'PANT FABRIC', 'BOTTOM WEIGHT'],
    'Dress Material': ['DRESS MATERIAL', 'DRESS FAB', 'LADIES FABRIC', 'KURTI', 'SALWAR', 'ANARKALI'],
    'Blouse Fabric': ['BLOUSE', 'BLOUSE PIECE', 'CHOLI'],
    'Denim': ['DENIM', 'JEANS', 'JEAN FABRIC', 'INDIGO', 'STRETCH DENIM'],
    # FINE COTTON
    'Voile': ['VOILE', 'COTTON VOILE'],
    'Cambric': ['CAMBRIC', 'LAWN', 'SWISS COTTON'],
    'Muslin': ['MUSLIN', 'MULMUL', 'MULL', 'JACONET'],
    'Poplin': ['POPLIN', 'BROADCLOTH'],
    'Organdi': ['ORGANDI', 'ORGANDY', 'ORGANZA'],
    # HOME TEXTILES
    'Sheeting': ['SHEETING', 'SHEET FABRIC', 'BED SHEET', 'BEDSHEET', 'BED LINEN'],
    'Furnishing': ['FURNISHING', 'FURNISH', 'UPHOLSTERY', 'SOFA FABRIC', 'CURTAIN', 'DRAPE'],
    'Terry Fabric': ['TERRY', 'TOWEL FABRIC', 'TOWELING', 'BATH TOWEL'],
    # INDUSTRIAL
    'Canvas': ['CANVAS', 'DUCK', 'HEAVY DUCK'],
    'Drill': ['DRILL', 'KHAKI DRILL', 'TWILL DRILL'],
    # WEAVE AS PRODUCT
    'Dobby Fabric': ['DOBBY', 'DOBBIE'],
    'Jacquard Fabric': ['JACQUARD', 'BROCADE', 'DAMASK'],
    'Oxford': ['OXFORD', 'BASKET WEAVE'],
    'Flannel': ['FLANNEL', 'FLANNELETTE', 'BRUSHED'],
    # HANDLOOM
    'Handloom': ['HANDLOOM', 'HAND LOOM', 'HAND WOVEN', 'HANDWOVEN', 'KHADI', 'KHADDAR'],
    # RELIGIOUS
    'Pooja Cloth': ['POOJA CLOTH', 'PUJA CLOTH', 'TEMPLE CLOTH', 'ASAN', 'AASAN'],
    # GREY/FINISHED
    'Grey Fabric': ['GREY FABRIC', 'GREIGE', 'RFD', 'PFD', 'LOOM STATE'],
}

PROCESSING_KEYWORDS = {
    'Raw': ['RAW', 'UNPROCESSED', 'NOT CARDED', 'NOT COMBED'],
    'Grey/Greige': ['GREY', 'GREIGE', 'GRAY', 'LOOM STATE', 'RFD', 'PFD', 'READY FOR DYE'],
    'Bleached': ['BLEACHED', 'WHITE', 'OPTICAL WHITE', 'OBA', 'SEMI BLEACH', 'FULL BLEACH'],
    'Dyed': ['DYED', 'SOLID DYED', 'PIECE DYED', 'VAT DYED', 'REACTIVE DYED', 'PIGMENT DYED',
             'SULPHUR DYED', 'COLORED', 'COLOURED'],
    'Yarn Dyed': ['YARN DYED', 'Y/D', 'YD', 'SPACE DYED', 'STRIPE', 'CHECK', 'PLAID', 'GINGHAM', 'CHAMBRAY'],
    'Printed': ['PRINTED', 'PRINT', 'SCREEN PRINT', 'ROTARY PRINT', 'DIGITAL PRINT', 'BLOCK PRINT'],
    'Mercerised': ['MERCERISED', 'MERCERIZED'],
    'Sanforised': ['SANFORISED', 'SANFORIZED', 'PRE SHRUNK', 'PRESHRUNK'],
    'Calendered': ['CALENDERED', 'GLAZED', 'CHINTZ'],
    'Enzyme Washed': ['ENZYME', 'BIO WASH', 'STONE WASH', 'ACID WASH'],
    'Waxed': ['WAXED', 'WAX FINISH'],
    'Unwaxed': ['UNWAXED', 'UN-WAXED'],
}

WEAVE_KEYWORDS = {
    'Plain': ['PLAIN', 'PLAIN WEAVE', 'TABBY', '1/1'],
    'Twill': ['TWILL', '2/1', '3/1', '2/2', 'DIAGONAL', 'HERRINGBONE'],
    'Satin': ['SATIN', 'SATEEN'],
    'Dobby': ['DOBBY', 'DOBBIE'],
    'Jacquard': ['JACQUARD', 'BROCADE', 'DAMASK'],
    'Oxford': ['OXFORD', 'BASKET', 'PANAMA'],
    'Denim': ['DENIM', '3/1 TWILL', 'RIGHT HAND TWILL'],
    'Terry': ['TERRY', 'LOOP PILE'],
    'Pique': ['PIQUE', 'WAFFLE', 'HONEYCOMB'],
    'Crepe': ['CREPE', 'CRINKLE', 'SEERSUCKER'],
    'Rib': ['RIB', 'CORD', 'BEDFORD'],
    'Leno': ['LENO', 'GAUZE'],
}

MATERIAL_KEYWORDS = {
    '100% Cotton': ['100% COTTON', '100 COTTON', '100PCT COTTON', 'PURE COTTON', 'ALL COTTON'],
    'Cotton-Polyester': ['COTTON POLY', 'POLY COTTON', 'PC', 'CVC', 'TC', 'POLYCOTTON', 'POLYESTER COTTON'],
    'Cotton-Viscose': ['COTTON VISCOSE', 'VISCOSE COTTON', 'CV', 'COTTON RAYON'],
    'Cotton-Linen': ['COTTON LINEN', 'LINEN COTTON'],
    'Cotton-Modal': ['COTTON MODAL', 'MODAL COTTON'],
    'Cotton-Lycra': ['COTTON LYCRA', 'COTTON SPANDEX', 'COTTON ELASTANE', 'STRETCH COTTON', 'LYCRA'],
    'Organic Cotton': ['ORGANIC', 'ORGANIC COTTON', 'ORG COTTON'],
    'BCI Cotton': ['BCI', 'BETTER COTTON'],
    'Recycled Cotton': ['RECYCLED', 'RECYCLE', 'REGEN', 'POST CONSUMER', 'GRS', 'RCS'],
}

CERTIFICATION_KEYWORDS = {
    'GOTS': ['GOTS', 'GLOBAL ORGANIC'],
    'OCS': ['OCS', 'ORGANIC CONTENT'],
    'NPOP': ['NPOP', 'NOP', 'USDA ORGANIC'],
    'BCI': ['BCI', 'BETTER COTTON'],
    'CmiA': ['CMIA', 'COTTON MADE IN AFRICA'],
    'Fairtrade': ['FAIRTRADE', 'FAIR TRADE'],
    'OEKO-TEX': ['OEKO-TEX', 'OEKOTEX', 'STANDARD 100'],
    'GRS': ['GRS', 'GLOBAL RECYCLE'],
    'Supima': ['SUPIMA'],
    'Pima': ['PIMA', 'PIMA COTTON'],
    'Egyptian': ['EGYPTIAN', 'GIZA'],
}

END_USE_KEYWORDS = {
    'Formal Apparel': ['FORMAL', 'OFFICE WEAR', 'BUSINESS'],
    'Casual Apparel': ['CASUAL', 'T-SHIRT', 'POLO', 'JEANS'],
    'Ethnic Wear': ['ETHNIC', 'TRADITIONAL', 'INDIAN WEAR'],
    'Kidswear': ['KIDS', 'CHILDREN', 'BABY', 'INFANT'],
    'Sportswear': ['SPORTS', 'ATHLETIC', 'ACTIVEWEAR', 'GYM'],
    'Sleepwear': ['SLEEP', 'NIGHT', 'PYJAMA', 'NIGHTWEAR'],
    'Home Textile': ['HOME', 'BEDDING', 'CURTAIN', 'FURNISHING', 'TOWEL'],
    'Industrial': ['INDUSTRIAL', 'WORKWEAR', 'UNIFORM'],
    'Religious': ['POOJA', 'PUJA', 'RELIGIOUS', 'TEMPLE'],
    'Knitting': ['FOR KNITTING', 'KNIT USE'],
    'Weaving': ['FOR WEAVING', 'LOOM USE'],
}

MAIN_CATEGORIES = {
    "5201": "Raw Cotton", "5202": "Cotton Waste", "5203": "Processed Cotton",
    "5204": "Cotton Thread", "5205": "Cotton Yarn (>=85%)", "5206": "Blended Yarn (<85%)",
    "5207": "Retail Yarn", "5208": "Light Fabric (<=200 GSM, >=85%)",
    "5209": "Heavy Fabric (>200 GSM, >=85%)", "5210": "Light Blended (<=200 GSM, <85%)",
    "5211": "Heavy Blended (>200 GSM, <85%)", "5212": "Other Woven"
}

GSM_CATS = {'Very Light (<100)':(0,100), 'Light (100-150)':(100,150), 'Medium (150-200)':(150,200),
            'Medium Heavy (200-300)':(200,300), 'Heavy (300-400)':(300,400), 'Very Heavy (>400)':(400,9999)}
YARN_CATS = {'Coarse (<=20s)':(0,20), 'Medium (20-40s)':(20,40), 'Fine (40-60s)':(40,60),
             'Superfine (60-80s)':(60,80), 'Ultrafine (>80s)':(80,999)}

print("✅ Dictionaries loaded!")

# CELL 4: LOAD & MERGE
print("\n📊 Loading files...")
data = pd.read_excel(DATA_FILE) if DATA_FILE.endswith(('.xlsx','.xls')) else pd.read_csv(DATA_FILE)
hsn = pd.read_excel(HSN_FILE) if HSN_FILE.endswith(('.xlsx','.xls')) else pd.read_csv(HSN_FILE)

data['HS_CODE'] = data['HS CODE'].astype(str).str.strip().str.zfill(8)
hsn['HSN_CD'] = hsn['HSN_CD'].astype(str).str.strip().str.zfill(8)
data = data.merge(hsn[['HSN_CD','HSN_Description']], left_on='HS_CODE', right_on='HSN_CD', how='left')
data['Heading'] = data['HS_CODE'].str[:4]
data['Main_Category'] = data['Heading'].map(MAIN_CATEGORIES).fillna('Other')

print(f"✅ Loaded {len(data):,} records")

# CELL 5: EXTRACTION FUNCTIONS
NOISE = [
    (r'\bAS\s*PER\s*(?:INVOICE|INV|PI|PL|LC|CONTRACT)\b',''), (r'\bOTHER\s*DETAILS?\b',''),
    (r'\bMADE\s*IN\s*[A-Z]+\b',''), (r'\b(?:FOB|CIF|CNF)\b',''), (r'\bSAMPLE\b',''),
    (r'\b(?:PO|INV|REF|LOT|STYLE)[:\s#]*[A-Z0-9\-]+\b',''), (r'\bGST[:\s]*[A-Z0-9]+\b',''),
    (r'\b\d{1,2}[\-/\.]\d{1,2}[\-/\.]\d{2,4}\b',''), (r'\bAPI\b',''),
    (r'\bWE\s*(?:INT)?\.?\s*CL(?:AI)?M\.?\s*ROD(?:TEP)?\b',''), (r'\bAWB[:\s\-]*\d+\b',''),
    (r'\bCROP\s*(?:YEAR)?[:\s]*\d{4}[\-/]?\d*\b',''), (r'\bTAX\s*INV[^\n,]*\b',''),
]

def clean(t):
    if pd.isna(t): return ""
    t = str(t).upper()
    for p,r in NOISE: t = re.sub(p,r,t,flags=re.I)
    return re.sub(r'\s+',' ',re.sub(r'[^\w\s\.\,\-\/\%\(\)]',' ',t)).strip()

def get_gsm(t):
    m = re.search(r'\b(\d{2,3})\s*(?:G\.?S\.?M\.?|GSM|G/?M2?)\b', str(t).upper())
    if m and 30<=int(m.group(1))<=600: return int(m.group(1))
    return None

def get_width(t):
    t = str(t).upper()
    m = re.search(r'\b(\d{2,3})\s*(?:INCH|IN|")\b', t)
    if m and 30<=int(m.group(1))<=150: return f"{m.group(1)} inch"
    m = re.search(r'\b(\d{2,3})\s*(?:CM|CMS)\b', t)
    if m and 50<=int(m.group(1))<=400: return f"{m.group(1)} cm"
    return None

def get_yarn_count(t):
    m = re.search(r'\bNE\s*[:\-]?\s*(\d+(?:/\d+)?)\b', str(t).upper())
    return f"Ne {m.group(1)}" if m else None

def get_thread_count(t):
    m = re.search(r'\b(\d{2,3})\s*[xX×]\s*(\d{2,3})\b', str(t).upper())
    return f"{m.group(1)}x{m.group(2)}" if m else None

def get_pct(t, kws):
    for kw in kws:
        m = re.search(rf'\b(\d{{1,3}})\s*%?\s*(?:PCT\s*)?{kw}\b', str(t).upper())
        if m and 1<=int(m.group(1))<=100: return int(m.group(1))
    return None

def match_kw(t, kw_dict, all_matches=False):
    t = str(t).upper()
    matches = []
    for cat, kws in kw_dict.items():
        if any(kw in t for kw in kws):
            if all_matches: matches.append(cat)
            else: return cat
    return list(set(matches)) if all_matches else None

def categorize(val, cats):
    if val is None: return None
    for c,(lo,hi) in cats.items():
        if lo<val<=hi: return c
    return None

def extract(row):
    t = row.get('PRODUCT DESCRIPTION','')
    h = row.get('Heading','')

    gsm = get_gsm(t)
    width = get_width(t)
    yarn = get_yarn_count(t)
    thread = get_thread_count(t)
    cot_pct = get_pct(t, ['COTTON'])
    poly_pct = get_pct(t, ['POLY','POLYESTER'])
    ela_pct = get_pct(t, ['ELASTANE','SPANDEX','LYCRA'])

    prod = match_kw(t, PRODUCT_KEYWORDS)
    if not prod:
        prod = {'5201':'Raw Cotton','5202':'Cotton Waste','5203':'Processed Fiber',
                '5204':'Thread','5205':'Cotton Yarn','5206':'Blended Yarn','5207':'Retail Yarn',
                '5208':'Light Fabric','5209':'Heavy Fabric','5210':'Light Blended',
                '5211':'Heavy Blended','5212':'Other Fabric'}.get(h,'Cotton Product')

    proc = match_kw(t, PROCESSING_KEYWORDS)
    if not proc and h=='5201': proc = 'Raw'

    weave = match_kw(t, WEAVE_KEYWORDS) if h in ['5208','5209','5210','5211','5212'] else None
    mat = match_kw(t, MATERIAL_KEYWORDS)
    certs = match_kw(t, CERTIFICATION_KEYWORDS, True)
    end = match_kw(t, END_USE_KEYWORDS)

    # Material composition string
    if cot_pct and poly_pct:
        mat_comp = f"{cot_pct}% Cotton {poly_pct}% Polyester"
        if ela_pct: mat_comp += f" {ela_pct}% Elastane"
    elif cot_pct:
        mat_comp = f"{cot_pct}% Cotton" + (f" {ela_pct}% Elastane" if ela_pct else "")
    else:
        mat_comp = mat

    yarn_num = int(re.search(r'(\d+)',yarn).group(1)) if yarn else None

    return {
        'Cleaned': clean(t), 'Product_Type': prod, 'Processing': proc, 'Weave': weave,
        'Material': mat_comp, 'End_Use': end, 'GSM': gsm, 'GSM_Cat': categorize(gsm, GSM_CATS),
        'Width': width, 'Yarn_Count': yarn, 'Yarn_Cat': categorize(yarn_num, YARN_CATS),
        'Thread_Count': thread, 'Cotton%': cot_pct, 'Poly%': poly_pct, 'Elastane%': ela_pct,
        'Certifications': ' | '.join(certs) if certs else None,
    }

print("✅ Functions ready!")

# CELL 6: EXTRACT
print("\n🔍 Extracting...")
ext = data.apply(extract, axis=1).apply(pd.Series)
for c in ext.columns: data[c] = ext[c]

print(f"\n📊 RESULTS ({len(data):,} records):")
for c in ext.columns:
    n = data[c].notna().sum()
    print(f"   {c:<20}: {n:>6,} ({100*n/len(data):>5.1f}%)")

# CELL 7: SUMMARY
print("\n📈 PRODUCT TYPE (Top 15):")
print(data['Product_Type'].value_counts().head(15))
print("\n📈 PROCESSING:")
print(data['Processing'].value_counts())
print("\n📈 WEAVE:")
print(data['Weave'].value_counts())
print("\n📈 CERTIFICATIONS:")
if data['Certifications'].notna().any():
    print(data['Certifications'].dropna().str.split(' \\| ').explode().value_counts())

# CELL 8: SAVE
print("\n💾 Saving...")
cols = ['HS_CODE','HSN_Description','Heading','Main_Category','PRODUCT DESCRIPTION','Cleaned',
        'Product_Type','Processing','Weave','Material','End_Use','GSM','GSM_Cat','Width',
        'Yarn_Count','Yarn_Cat','Thread_Count','Cotton%','Poly%','Elastane%','Certifications']
cols = [c for c in cols if c in data.columns]
out = data[cols].rename(columns={'PRODUCT DESCRIPTION':'Original'})

ts = datetime.now().strftime('%Y%m%d_%H%M%S')
fn = f'/content/Chapter52_V4_{ts}.xlsx'
out.to_excel(fn, index=False)
print(f"✅ Saved: {fn}")

files.download(fn)
print("\n🎉 COMPLETE!")
print(f"   Total: {len(out):,} | Product: {out['Product_Type'].notna().sum():,} | GSM: {out['GSM'].notna().sum():,}")

In [ ]:
"""
================================================================================
EXPORT DATA ANALYZER V2 - COMPREHENSIVE STATISTICS & DATA CLEANING
================================================================================
Combined pipeline:
- Full data cleaning (error values, IEC matching, company standardization)
- FOB categorization & outlier detection
- Country standardization with fuzzy matching
- HSN Code statistics (chapters, headings, tariff items)
- Importer/Exporter analysis
- Country & Zone mapping (Europe, East Asia, etc.)
- Port analysis, Value & Volume metrics, Time-based trends

For Google Colab - Run each cell in order
================================================================================
"""

# ============================================================================
# CELL 1: INSTALL & IMPORT
# ============================================================================

!pip install -q pandas openpyxl xlsxwriter rapidfuzz

import pandas as pd
import numpy as np
import re
import json
from collections import defaultdict
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

try:
    from rapidfuzz import process, fuzz
    FUZZY_AVAILABLE = True
except:
    FUZZY_AVAILABLE = False
    print("⚠️ rapidfuzz not available - country fuzzy matching disabled")

from google.colab import files

print("✅ Dependencies installed!")
print(f"📅 Session: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# CELL 2: UPLOAD FILES
# ============================================================================

print("="*60)
print("📂 Upload EXPORT DATA FILE")
print("   Required columns: HS CODE, PRODUCT DESCRIPTION, EXPORTER NAME,")
print("   IMPORTER NAME, FOREIGN COUNTRY, FOB, QUANTITY, SB DATE, IEC, etc.")
print("="*60)
uploaded = files.upload()
DATA_FILE = '/content/' + list(uploaded.keys())[0]
print(f"✅ Data File: {DATA_FILE}")

print("\n📂 Upload COUNTRIES REFERENCE FILE (Optional - for country standardization)")
print("   Skip by pressing Cancel if not available")
try:
    uploaded_countries = files.upload()
    if uploaded_countries:
        COUNTRIES_REF = '/content/' + list(uploaded_countries.keys())[0]
        print(f"✅ Countries File: {COUNTRIES_REF}")
    else:
        COUNTRIES_REF = None
except:
    COUNTRIES_REF = None
    print("⚠️ No countries reference file - using basic standardization")

# ============================================================================
# CELL 3: CONFIGURATION & MAPPINGS
# ============================================================================

# Cleaning parameters
MIN_STANDARD_FOB = 10000  # Threshold for "standard" vs "sample" transactions
FOB_OUTLIER_PERCENTILES = (0.01, 0.99)  # Remove extreme outliers

# Error values to replace with NaN
ERROR_VALUES = [
    "#REF!", "#NAME?", "#N/A", "#VALUE!", "#DIV/0!", "#NULL!",
    "nana", "NA", "N/A", "Na", "N A", "N?A", "n/a",
    "Null", "NULL", "null", "UNKNOWN", "Unknown", "unknown",
    "NOT FOUND", "Not Found", "not found",
    "0000", "00000000", "000000",
    "NIL", "Nil", "nil", "NONE", "None", "none",
    "-", "--", "---", "____", "X", "XX", "XXX"
]

# Unit standardization
UNIT_CORRECTIONS = {
    'KGA': 'KGS', 'KILO': 'KGS', 'KG': 'KGS', 'KILOGRAMS': 'KGS',
    'MET': 'MTR', 'METER': 'MTR', 'METRES': 'MTR', 'METERS': 'MTR',
    'TONNE': 'MTS', 'TON': 'MTS', 'MT': 'MTS', 'TONS': 'MTS',
    'PIECE': 'PCS', 'PC': 'PCS', 'PIECES': 'PCS',
    'NUMBER': 'NOS', 'NO': 'NOS', 'NUM': 'NOS', 'NUMBERS': 'NOS',
    'DOZEN': 'DOZ', 'DZ': 'DOZ', 'PAIR': 'PRS', 'PAIRS': 'PRS',
    'SETS': 'SET', 'SQM': 'SQM', 'SQMTR': 'SQM', 'SQF': 'SQF', 'SQFT': 'SQF',
    'GROSS': 'GRS', 'FEET': 'FTS', 'FT': 'FTS',
    'CENTIMETER': 'CMS', 'CM': 'CMS', 'KILOMETER': 'KME', 'KM': 'KME',
    'CARTON': 'CTN', 'CTNS': 'CTN', 'LITER': 'LTR', 'LITRE': 'LTR', 'LTS': 'LTR'
}

# Geographic Zone Mapping
COUNTRY_ZONES = {
    # EUROPE
    'UNITED KINGDOM': 'Europe', 'UK': 'Europe', 'GREAT BRITAIN': 'Europe',
    'GERMANY': 'Europe', 'FRANCE': 'Europe', 'ITALY': 'Europe', 'SPAIN': 'Europe',
    'NETHERLANDS': 'Europe', 'BELGIUM': 'Europe', 'POLAND': 'Europe',
    'SWEDEN': 'Europe', 'AUSTRIA': 'Europe', 'SWITZERLAND': 'Europe',
    'DENMARK': 'Europe', 'FINLAND': 'Europe', 'NORWAY': 'Europe',
    'IRELAND': 'Europe', 'PORTUGAL': 'Europe', 'GREECE': 'Europe',
    'CZECH REPUBLIC': 'Europe', 'ROMANIA': 'Europe', 'HUNGARY': 'Europe',
    'SLOVAKIA': 'Europe', 'BULGARIA': 'Europe', 'CROATIA': 'Europe',
    'SLOVENIA': 'Europe', 'LITHUANIA': 'Europe', 'LATVIA': 'Europe',
    'ESTONIA': 'Europe', 'LUXEMBOURG': 'Europe', 'MALTA': 'Europe',
    'CYPRUS': 'Europe', 'ICELAND': 'Europe',

    # NORTH AMERICA
    'UNITED STATES': 'North America', 'USA': 'North America', 'U.S.A.': 'North America',
    'UNITED STATES OF AMERICA': 'North America', 'US': 'North America',
    'CANADA': 'North America', 'MEXICO': 'North America',

    # EAST ASIA
    'CHINA': 'East Asia', 'JAPAN': 'East Asia', 'SOUTH KOREA': 'East Asia',
    'KOREA': 'East Asia', 'REPUBLIC OF KOREA': 'East Asia',
    'TAIWAN': 'East Asia', 'HONG KONG': 'East Asia', 'MACAU': 'East Asia',
    'MONGOLIA': 'East Asia',

    # SOUTHEAST ASIA
    'SINGAPORE': 'Southeast Asia', 'MALAYSIA': 'Southeast Asia',
    'THAILAND': 'Southeast Asia', 'VIETNAM': 'Southeast Asia', 'VIET NAM': 'Southeast Asia',
    'INDONESIA': 'Southeast Asia', 'PHILIPPINES': 'Southeast Asia',
    'MYANMAR': 'Southeast Asia', 'CAMBODIA': 'Southeast Asia',
    'LAOS': 'Southeast Asia', 'BRUNEI': 'Southeast Asia',

    # SOUTH ASIA
    'BANGLADESH': 'South Asia', 'SRI LANKA': 'South Asia',
    'PAKISTAN': 'South Asia', 'NEPAL': 'South Asia',
    'BHUTAN': 'South Asia', 'MALDIVES': 'South Asia', 'AFGHANISTAN': 'South Asia',

    # MIDDLE EAST
    'UNITED ARAB EMIRATES': 'Middle East', 'UAE': 'Middle East', 'DUBAI': 'Middle East',
    'SAUDI ARABIA': 'Middle East', 'QATAR': 'Middle East', 'KUWAIT': 'Middle East',
    'BAHRAIN': 'Middle East', 'OMAN': 'Middle East', 'JORDAN': 'Middle East',
    'LEBANON': 'Middle East', 'ISRAEL': 'Middle East', 'IRAQ': 'Middle East',
    'IRAN': 'Middle East', 'YEMEN': 'Middle East', 'SYRIA': 'Middle East',

    # AFRICA
    'SOUTH AFRICA': 'Africa', 'EGYPT': 'Africa', 'NIGERIA': 'Africa',
    'KENYA': 'Africa', 'MOROCCO': 'Africa', 'ETHIOPIA': 'Africa',
    'GHANA': 'Africa', 'TANZANIA': 'Africa', 'UGANDA': 'Africa',
    'ALGERIA': 'Africa', 'SUDAN': 'Africa', 'TUNISIA': 'Africa',
    'MAURITIUS': 'Africa', 'SENEGAL': 'Africa', 'IVORY COAST': 'Africa',
    'CAMEROON': 'Africa', 'ZIMBABWE': 'Africa', 'MOZAMBIQUE': 'Africa',
    'ANGOLA': 'Africa', 'LIBYA': 'Africa', 'ZAMBIA': 'Africa',
    'BOTSWANA': 'Africa', 'NAMIBIA': 'Africa', 'MADAGASCAR': 'Africa',

    # SOUTH AMERICA
    'BRAZIL': 'South America', 'ARGENTINA': 'South America',
    'CHILE': 'South America', 'COLOMBIA': 'South America',
    'PERU': 'South America', 'VENEZUELA': 'South America',
    'ECUADOR': 'South America', 'BOLIVIA': 'South America',
    'PARAGUAY': 'South America', 'URUGUAY': 'South America',

    # CENTRAL AMERICA & CARIBBEAN
    'PANAMA': 'Central America & Caribbean', 'COSTA RICA': 'Central America & Caribbean',
    'GUATEMALA': 'Central America & Caribbean', 'HONDURAS': 'Central America & Caribbean',
    'JAMAICA': 'Central America & Caribbean', 'CUBA': 'Central America & Caribbean',
    'DOMINICAN REPUBLIC': 'Central America & Caribbean', 'HAITI': 'Central America & Caribbean',
    'TRINIDAD': 'Central America & Caribbean', 'BAHAMAS': 'Central America & Caribbean',

    # OCEANIA
    'AUSTRALIA': 'Oceania', 'NEW ZEALAND': 'Oceania',
    'FIJI': 'Oceania', 'PAPUA NEW GUINEA': 'Oceania',

    # CIS / CENTRAL ASIA
    'RUSSIA': 'CIS', 'RUSSIAN FEDERATION': 'CIS', 'UKRAINE': 'CIS',
    'KAZAKHSTAN': 'CIS', 'UZBEKISTAN': 'CIS', 'TURKMENISTAN': 'CIS',
    'AZERBAIJAN': 'CIS', 'GEORGIA': 'CIS', 'ARMENIA': 'CIS',
    'BELARUS': 'CIS', 'MOLDOVA': 'CIS', 'KYRGYZSTAN': 'CIS', 'TAJIKISTAN': 'CIS',

    # TURKEY
    'TURKEY': 'Turkey/Eurasia', 'TURKIYE': 'Turkey/Eurasia',
}

# Trade Bloc Mapping
TRADE_BLOCS = {
    # EU Members
    'GERMANY': 'EU', 'FRANCE': 'EU', 'ITALY': 'EU', 'SPAIN': 'EU',
    'NETHERLANDS': 'EU', 'BELGIUM': 'EU', 'POLAND': 'EU', 'SWEDEN': 'EU',
    'AUSTRIA': 'EU', 'DENMARK': 'EU', 'FINLAND': 'EU', 'IRELAND': 'EU',
    'PORTUGAL': 'EU', 'GREECE': 'EU', 'CZECH REPUBLIC': 'EU', 'ROMANIA': 'EU',
    'HUNGARY': 'EU', 'SLOVAKIA': 'EU', 'BULGARIA': 'EU', 'CROATIA': 'EU',
    'SLOVENIA': 'EU', 'LITHUANIA': 'EU', 'LATVIA': 'EU', 'ESTONIA': 'EU',
    'LUXEMBOURG': 'EU', 'MALTA': 'EU', 'CYPRUS': 'EU',

    # ASEAN
    'SINGAPORE': 'ASEAN', 'MALAYSIA': 'ASEAN', 'THAILAND': 'ASEAN',
    'VIETNAM': 'ASEAN', 'VIET NAM': 'ASEAN', 'INDONESIA': 'ASEAN',
    'PHILIPPINES': 'ASEAN', 'MYANMAR': 'ASEAN', 'CAMBODIA': 'ASEAN',
    'LAOS': 'ASEAN', 'BRUNEI': 'ASEAN',

    # GCC
    'UNITED ARAB EMIRATES': 'GCC', 'UAE': 'GCC', 'SAUDI ARABIA': 'GCC',
    'QATAR': 'GCC', 'KUWAIT': 'GCC', 'BAHRAIN': 'GCC', 'OMAN': 'GCC',

    # SAARC
    'BANGLADESH': 'SAARC', 'SRI LANKA': 'SAARC', 'PAKISTAN': 'SAARC',
    'NEPAL': 'SAARC', 'BHUTAN': 'SAARC', 'MALDIVES': 'SAARC', 'AFGHANISTAN': 'SAARC',

    # USMCA
    'UNITED STATES': 'USMCA', 'USA': 'USMCA', 'CANADA': 'USMCA', 'MEXICO': 'USMCA',

    # MERCOSUR
    'BRAZIL': 'MERCOSUR', 'ARGENTINA': 'MERCOSUR', 'PARAGUAY': 'MERCOSUR', 'URUGUAY': 'MERCOSUR',

    # AfCFTA
    'SOUTH AFRICA': 'AfCFTA', 'EGYPT': 'AfCFTA', 'NIGERIA': 'AfCFTA', 'KENYA': 'AfCFTA',
    'MOROCCO': 'AfCFTA', 'ETHIOPIA': 'AfCFTA', 'GHANA': 'AfCFTA',
}

# HSN Chapter Names
HSN_CHAPTERS = {
    '52': 'Cotton', '54': 'Man-made Filaments', '55': 'Man-made Staple Fibres',
    '58': 'Special Woven Fabrics', '60': 'Knitted Fabrics', '61': 'Knitted Apparel',
    '62': 'Woven Apparel', '63': 'Made-up Textile Articles',
    '30': 'Pharmaceutical Products', '68': 'Stone, Plaster, Cement',
    '73': 'Iron & Steel Articles', '84': 'Machinery', '85': 'Electrical Equipment',
    '87': 'Vehicles', '90': 'Optical/Medical Instruments',
    '39': 'Plastics', '40': 'Rubber', '72': 'Iron & Steel',
    '29': 'Organic Chemicals', '28': 'Inorganic Chemicals',
}

# Indian Port Mapping
INDIAN_PORTS = {
    'INNSA': 'Nhava Sheva (JNPT)', 'INMUN': 'Mundra', 'INCHE': 'Chennai',
    'INKOL': 'Kolkata', 'INBOM': 'Mumbai', 'INTUT': 'Tuticorin',
    'INCOK': 'Cochin', 'INPAV': 'Pipavav', 'INVTZ': 'Vizag',
    'INKRI': 'Krishnapatnam', 'INIXE': 'Goa', 'INMAA': 'Chennai Air',
    'INDEL': 'Delhi Air (IGI)', 'INBLR': 'Bangalore Air', 'INHYD': 'Hyderabad Air',
    'INAMD': 'Ahmedabad', 'INCCU': 'Kolkata Air', 'INPNQ': 'Pune',
}

# Manual country overrides for fuzzy matching
MANUAL_COUNTRY_MAP = {
    "united states": "United States of America", "u.s.a": "United States of America",
    "usa": "United States of America", "us": "United States of America",
    "america": "United States of America", "u.s": "United States of America",
    "uae": "United Arab Emirates", "u.a.e": "United Arab Emirates",
    "uk": "United Kingdom", "u.k": "United Kingdom", "great britain": "United Kingdom",
    "britain": "United Kingdom", "turkey": "Turkiye", "turkiye": "Turkiye",
    "korea republic of": "Republic of Korea", "south korea": "Republic of Korea",
    "s korea": "Republic of Korea", "vietnam": "Vietnam", "viet nam": "Vietnam",
    "russia": "Russian Federation", "russian federation": "Russian Federation",
    "iran": "Iran", "ivory coast": "Cote d'Ivoire", "unknown": None, "nan": None,
}

print("✅ Configuration & mappings loaded!")

# ============================================================================
# CELL 4: CLEANING FUNCTIONS
# ============================================================================

def standardize_company_suffix(name):
    """Standardize company legal suffixes"""
    if pd.isna(name):
        return name
    name = str(name).upper().strip()
    suffix_mappings = [
        (r'\bPVT\.?\s*LTD\.?\s*$', 'PRIVATE LIMITED'),
        (r'\bPRIVATE\s*LTD\.?\s*$', 'PRIVATE LIMITED'),
        (r'\bPRIVATE\s*LIMITED\s*$', 'PRIVATE LIMITED'),
        (r'\bP\.?\s*LTD\.?\s*$', 'PRIVATE LIMITED'),
        (r'\bLIMITED\s*$', 'LIMITED'), (r'\bLTD\.?\s*$', 'LIMITED'),
        (r'\bCORPORATION\s*$', 'CORPORATION'), (r'\bCORP\.?\s*$', 'CORPORATION'),
        (r'\bINCORPORATED\s*$', 'INCORPORATED'), (r'\bINC\.?\s*$', 'INCORPORATED'),
        (r'\bCOMPANY\s*$', 'COMPANY'), (r'\bCO\.?\s*$', 'COMPANY'),
        (r'\bL\.?L\.?C\.?\s*$', 'LLC'), (r'\bGMBH\s*$', 'GMBH'),
        (r'\bPVT\.?\s*$', 'PRIVATE LIMITED'), (r'\bPROP\.?\s*', 'PROPRIETOR '),
    ]
    for pattern, replacement in suffix_mappings:
        name = re.sub(pattern, replacement, name)
    return name

def clean_company_name(name):
    """Clean company name - remove extra spaces, special chars"""
    if pd.isna(name):
        return name
    name = str(name).upper().strip()
    name = re.sub(r'[^A-Z0-9\s&-]', ' ', name)
    name = re.sub(r'([A-Z])-([A-Z])', r'\1 \2', name)
    name = re.sub(r'\s+', ' ', name)
    return name.strip()

def remove_common_words(name):
    """Remove common filler words"""
    if pd.isna(name):
        return name
    name = str(name).upper()
    remove_patterns = [r'^THE\s+', r'^M/S\s+', r'^M\.?S\.?\s+', r'^MESSRS\.?\s+',
                       r'\s+&\s+ASSOCIATES$', r'\s+AND\s+CO$']
    for pattern in remove_patterns:
        name = re.sub(pattern, '', name)
    return name.strip()

def standardize_company_name_full(name):
    """Full company name standardization pipeline"""
    if pd.isna(name):
        return name
    name = clean_company_name(name)
    name = standardize_company_suffix(name)
    name = remove_common_words(name)
    return re.sub(r'\s+', ' ', name).strip()

def normalize_country(x):
    """Basic country normalization"""
    if pd.isna(x):
        return ""
    x = str(x).strip().upper()
    x = re.sub(r"[.,''-]", "", x)
    x = re.sub(r"\s+", " ", x)
    return x

print("✅ Cleaning functions loaded!")

# ============================================================================
# CELL 5: LOAD DATA & INITIAL CLEANING
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 1: LOADING & INITIAL CLEANING")
print("="*60)

# Load file
if DATA_FILE.endswith(('.xlsx', '.xls')):
    data_raw = pd.read_excel(DATA_FILE)
else:
    data_raw = pd.read_csv(DATA_FILE, encoding='utf-8', on_bad_lines='skip')

original_count = len(data_raw)
print(f"✅ Loaded {original_count:,} records")
print(f"📋 Columns: {list(data_raw.columns)}")

# Create working copy
data = data_raw.copy()

# Replace error values with NaN
data = data.replace(ERROR_VALUES, None)
for col in data.select_dtypes(include='object').columns:
    data[col] = data[col].apply(lambda x: None if isinstance(x, str) and x.strip() == '' else x)
print("✅ Replaced error values with NaN")

# Standardize column names
col_mapping = {
    'HS CODE': 'HS_CODE', 'HSCODE': 'HS_CODE', 'HS4': 'HS4',
    'PRODUCT DESCRIPTION': 'PRODUCT_DESC', 'PRODUCT_DESCRIPTION': 'PRODUCT_DESC',
    'EXPORTER NAME': 'EXPORTER', 'EXPORTER_NAME': 'EXPORTER',
    'IMPORTER NAME': 'IMPORTER', 'IMPORTER_NAME': 'IMPORTER', 'IMPORTER NAME ': 'IMPORTER',
    'FOREIGN COUNTRY': 'COUNTRY', 'FOREIGN_COUNTRY': 'COUNTRY',
    'FOREIGN PORT': 'FOREIGN_PORT', 'INDIAN PORT': 'INDIAN_PORT',
    'FOB': 'FOB_VALUE', 'FOB INR': 'FOB_VALUE', 'VALUE IN FC': 'FC_VALUE',
    'QUANTITY': 'QTY', 'UNIT QUANTITY': 'UNIT_QTY', 'QUANTITY UNIT': 'QTY_UNIT',
    'SB DATE': 'SB_DATE', 'SB NO': 'SB_NO', 'IEC': 'IEC',
}

for old, new in col_mapping.items():
    if old in data.columns and new not in data.columns:
        data[new] = data[old]

print("✅ Column names standardized")

# ============================================================================
# CELL 6: DATA TYPE CONVERSIONS
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 2: DATA TYPE CONVERSIONS")
print("="*60)

# Convert SB_DATE to datetime
if 'SB_DATE' in data.columns:
    data['SB_DATE'] = pd.to_datetime(data['SB_DATE'], errors='coerce')
    valid_dates = data['SB_DATE'].dropna()
    if len(valid_dates) > 0:
        print(f"✅ Date range: {valid_dates.min()} to {valid_dates.max()}")
    else:
        print("⚠️ No valid dates found")

# Convert FOB to numeric - handle commas, currency symbols, and various formats
if 'FOB_VALUE' in data.columns:
    # First convert to string
    data['FOB_VALUE'] = data['FOB_VALUE'].astype(str)
    # Remove commas, currency symbols, and common text
    data['FOB_VALUE'] = (
        data['FOB_VALUE']
        .str.strip()
        .str.replace(',', '', regex=False)  # Remove thousand separators
        .str.replace('₹', '', regex=False)
        .str.replace('Rs', '', regex=False)
        .str.replace('Rs.', '', regex=False)
        .str.replace('INR', '', regex=False)
        .str.replace('USD', '', regex=False)
        .str.replace('$', '', regex=False)
        .str.replace(' ', '', regex=False)
    )
    # Replace invalid values
    data['FOB_VALUE'] = data['FOB_VALUE'].replace(['None', 'nan', 'NaN', 'NA', '', '-'], '0')
    # Convert to numeric
    data['FOB_VALUE'] = pd.to_numeric(data['FOB_VALUE'], errors='coerce').fillna(0)
    print(f"✅ FOB range: ₹{data['FOB_VALUE'].min():,.0f} to ₹{data['FOB_VALUE'].max():,.0f}")

# Convert Quantity to numeric
if 'QTY' in data.columns:
    data['QTY'] = pd.to_numeric(data['QTY'], errors='coerce')
    print(f"✅ Quantity range: {data['QTY'].min():,.0f} to {data['QTY'].max():,.0f}")

# Clean HS Code
if 'HS_CODE' in data.columns:
    data['HS_CODE'] = data['HS_CODE'].astype(str).str.strip().str.replace('.0','',regex=False)
    data['HS_CODE'] = data['HS_CODE'].str.zfill(8)
    data['HS4'] = data['HS_CODE'].str[:4]
    data['HS6'] = data['HS_CODE'].str[:6]
    data['Chapter'] = data['HS_CODE'].str[:2]
    print(f"✅ HS Codes cleaned: {data['HS_CODE'].nunique():,} unique")
elif 'HS4' in data.columns:
    data['HS4'] = data['HS4'].astype(str).str.strip().str.zfill(4)
    data['Chapter'] = data['HS4'].str[:2]

# Map chapter names
if 'Chapter' in data.columns:
    data['Chapter_Name'] = data['Chapter'].map(HSN_CHAPTERS).fillna('Other')

# ============================================================================
# CELL 7: REMOVE RECORDS WITH CRITICAL NULLS
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 3: REMOVING CRITICAL NULLS")
print("="*60)

before_null_filter = len(data)

# Define critical columns (check which exist)
critical_checks = []
if 'SB_DATE' in data.columns:
    critical_checks.append(data['SB_DATE'].notna())
if 'FOB_VALUE' in data.columns:
    critical_checks.append(data['FOB_VALUE'].notna())
if 'EXPORTER' in data.columns:
    critical_checks.append(data['EXPORTER'].notna())

if critical_checks:
    combined_filter = critical_checks[0]
    for check in critical_checks[1:]:
        combined_filter = combined_filter & check
    data = data[combined_filter]

removed_nulls = before_null_filter - len(data)
print(f"✅ Removed {removed_nulls:,} records with critical nulls ({removed_nulls/before_null_filter*100:.2f}%)")
print(f"   Remaining: {len(data):,} records")

# ============================================================================
# CELL 8: IEC MATCHING & EXPORTER ID
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 4: IEC MATCHING")
print("="*60)

if 'IEC' in data.columns and 'EXPORTER' in data.columns:
    # Find exporters with at least one valid IEC
    exporter_iec_map = (
        data[data['IEC'].notna()]
        .groupby('EXPORTER')['IEC']
        .first()
        .to_dict()
    )

    print(f"📊 IEC Statistics:")
    print(f"   Exporters with at least 1 IEC: {len(exporter_iec_map):,}")
    print(f"   Records with original IEC: {data['IEC'].notna().sum():,}")
    print(f"   Records missing IEC: {data['IEC'].isna().sum():,}")

    # Fill missing IEC using exporter name match
    data['IEC_FILLED'] = data.apply(
        lambda row: exporter_iec_map.get(row['EXPORTER']) if pd.isna(row['IEC']) else row['IEC'],
        axis=1
    )

    # Create IEC status flag
    data['IEC_STATUS'] = data.apply(
        lambda row: 'ORIGINAL' if pd.notna(row['IEC'])
                    else ('MATCHED' if pd.notna(row['IEC_FILLED']) else 'NO_IEC'),
        axis=1
    )

    # Create primary exporter ID
    data['EXPORTER_ID'] = data['IEC_FILLED']

    print(f"\n✅ IEC Matching Complete:")
    print(f"   Original IEC: {(data['IEC_STATUS'] == 'ORIGINAL').sum():,}")
    print(f"   Matched IEC: {(data['IEC_STATUS'] == 'MATCHED').sum():,}")
    print(f"   No IEC: {(data['IEC_STATUS'] == 'NO_IEC').sum():,}")
else:
    print("⚠️ IEC or EXPORTER column not found - skipping IEC matching")
    if 'EXPORTER' in data.columns:
        data['EXPORTER_ID'] = data['EXPORTER']

# ============================================================================
# CELL 9: COMPANY NAME STANDARDIZATION
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 5: COMPANY NAME STANDARDIZATION")
print("="*60)

if 'EXPORTER' in data.columns:
    data['EXPORTER_ORIGINAL'] = data['EXPORTER']
    before_exp = data['EXPORTER'].nunique()
    data['EXPORTER'] = data['EXPORTER'].apply(standardize_company_name_full)
    after_exp = data['EXPORTER'].nunique()
    print(f"✅ Exporters: {before_exp:,} → {after_exp:,} unique (consolidated {before_exp - after_exp:,})")

if 'IMPORTER' in data.columns:
    data['IMPORTER_ORIGINAL'] = data['IMPORTER']
    before_imp = data['IMPORTER'].nunique()
    data['IMPORTER'] = data['IMPORTER'].apply(standardize_company_name_full)
    after_imp = data['IMPORTER'].nunique()
    print(f"✅ Importers: {before_imp:,} → {after_imp:,} unique (consolidated {before_imp - after_imp:,})")

# ============================================================================
# CELL 10: FOB CATEGORIZATION & OUTLIER DETECTION
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 6: FOB CATEGORIZATION")
print("="*60)

if 'FOB_VALUE' in data.columns:
    # Create FOB categories
    data['FOB_CATEGORY'] = pd.cut(
        data['FOB_VALUE'],
        bins=[-np.inf, 0, 1000, MIN_STANDARD_FOB, np.inf],
        labels=['ZERO', 'SAMPLE', 'LOW_VALUE', 'STANDARD']
    )

    # Create boolean flags
    data['IS_ZERO_FOB'] = data['FOB_VALUE'] == 0
    data['IS_SAMPLE'] = data['FOB_VALUE'].between(1, 1000)
    data['IS_LOW_VALUE'] = data['FOB_VALUE'].between(1001, MIN_STANDARD_FOB - 1)
    data['IS_STANDARD'] = data['FOB_VALUE'] >= MIN_STANDARD_FOB

    print("📊 FOB Distribution:")
    print(data['FOB_CATEGORY'].value_counts().sort_index())

    # Calculate FOB per unit and detect outliers
    if 'QTY' in data.columns:
        data['FOB_PER_UNIT'] = data['FOB_VALUE'] / data['QTY'].replace(0, np.nan)

        q_low, q_high = FOB_OUTLIER_PERCENTILES
        percentile_low = data['FOB_PER_UNIT'].quantile(q_low)
        percentile_high = data['FOB_PER_UNIT'].quantile(q_high)

        data['IS_FOB_OUTLIER'] = (
            (data['FOB_PER_UNIT'] > percentile_high) |
            (data['FOB_PER_UNIT'] < percentile_low)
        )

        outliers = data['IS_FOB_OUTLIER'].sum()
        print(f"\n📊 FOB Outliers: {outliers:,} ({outliers/len(data)*100:.2f}%)")
    else:
        data['IS_FOB_OUTLIER'] = False

# ============================================================================
# CELL 11: UNIT STANDARDIZATION
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 7: UNIT STANDARDIZATION")
print("="*60)

if 'QTY_UNIT' in data.columns:
    data['QTY_UNIT'] = data['QTY_UNIT'].str.strip().str.upper().replace(UNIT_CORRECTIONS)
    print("📦 Unit Distribution (Top 10):")
    print(data['QTY_UNIT'].value_counts().head(10))
    print(f"   Total unique units: {data['QTY_UNIT'].nunique()}")

# ============================================================================
# CELL 12: COUNTRY STANDARDIZATION & ZONE MAPPING
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 8: COUNTRY STANDARDIZATION")
print("="*60)

if 'COUNTRY' in data.columns:
    data['COUNTRY'] = data['COUNTRY'].astype(str).str.upper().str.strip()
    data['COUNTRY_ORIGINAL'] = data['COUNTRY']

    # Apply manual overrides first
    def apply_country_mapping(country):
        country_lower = str(country).lower().strip()
        if country_lower in MANUAL_COUNTRY_MAP:
            return MANUAL_COUNTRY_MAP[country_lower] or country
        return country

    data['COUNTRY_CLEANED'] = data['COUNTRY'].apply(apply_country_mapping)

    # Map zones and trade blocs
    data['Zone'] = data['COUNTRY'].map(COUNTRY_ZONES).fillna('Other')
    data['Trade_Bloc'] = data['COUNTRY'].map(TRADE_BLOCS).fillna('Non-bloc')

    print(f"✅ Unique countries: {data['COUNTRY'].nunique()}")
    print(f"✅ Zones mapped: {data['Zone'].nunique()}")
    print(f"✅ Trade blocs mapped: {data[data['Trade_Bloc'] != 'Non-bloc']['Trade_Bloc'].nunique()}")

# ============================================================================
# CELL 13: TIME FEATURES
# ============================================================================

print("\n" + "="*60)
print("📊 STAGE 9: TIME FEATURES")
print("="*60)

if 'SB_DATE' in data.columns:
    valid_dates = data['SB_DATE'].dropna()

    if len(valid_dates) > 0:
        data['YEAR'] = data['SB_DATE'].dt.year
        data['MONTH'] = data['SB_DATE'].dt.month
        data['QUARTER'] = data['SB_DATE'].dt.quarter
        data['DAY_OF_WEEK'] = data['SB_DATE'].dt.weekday
        data['DAY_OF_MONTH'] = data['SB_DATE'].dt.day
        data['WEEK_OF_YEAR'] = data['SB_DATE'].dt.isocalendar().week.astype(int)
        data['YearMonth'] = data['SB_DATE'].dt.to_period('M').astype(str)

        print(f"✅ Time features created")
        print(f"   Date range: {valid_dates.min().strftime('%Y-%m-%d')} to {valid_dates.max().strftime('%Y-%m-%d')}")
        print(f"   Years: {sorted(data['YEAR'].dropna().unique().astype(int))}")
        print(f"   Months: {sorted(data['MONTH'].dropna().unique().astype(int))}")

# ============================================================================
# CELL 14: HSN CODE STATISTICS
# ============================================================================

print("\n" + "="*60)
print("📊 HSN CODE STATISTICS")
print("="*60)

# Chapter level
if 'Chapter' in data.columns:
    chapter_dist = data['Chapter'].value_counts()
    print(f"\n📌 Unique Chapters: {data['Chapter'].nunique()}")
    print("\n📈 TOP 10 CHAPTERS:")
    for ch, cnt in chapter_dist.head(10).items():
        ch_name = HSN_CHAPTERS.get(ch, 'Unknown')
        print(f"   {ch} - {ch_name}: {cnt:,} ({100*cnt/len(data):.1f}%)")

# HS4 level
if 'HS4' in data.columns:
    print(f"\n📌 Unique Headings (HS4): {data['HS4'].nunique()}")
    print("\n📈 TOP 15 HEADINGS (HS4):")
    for hs4, cnt in data['HS4'].value_counts().head(15).items():
        print(f"   {hs4}: {cnt:,} ({100*cnt/len(data):.1f}%)")

# HS8 level
if 'HS_CODE' in data.columns:
    print(f"\n📌 Unique Tariff Items (HS8): {data['HS_CODE'].nunique()}")

# ============================================================================
# CELL 15: IMPORTER & EXPORTER STATISTICS
# ============================================================================

print("\n" + "="*60)
print("👥 IMPORTER & EXPORTER STATISTICS")
print("="*60)

if 'EXPORTER' in data.columns:
    print(f"\n📌 Unique Exporters: {data['EXPORTER'].nunique():,}")
    print("\n📈 TOP 15 EXPORTERS (by shipments):")
    for exp, cnt in data['EXPORTER'].value_counts().head(15).items():
        print(f"   {exp[:50]}: {cnt:,}")

if 'IMPORTER' in data.columns:
    valid_importers = data[~data['IMPORTER'].isin(['', 'NAN', 'NONE', 'NA', '-', 'nan'])]
    print(f"\n📌 Unique Importers: {valid_importers['IMPORTER'].nunique():,}")
    print("\n📈 TOP 15 IMPORTERS (by shipments):")
    for imp, cnt in valid_importers['IMPORTER'].value_counts().head(15).items():
        print(f"   {imp[:50]}: {cnt:,}")

# ============================================================================
# CELL 16: COUNTRY & ZONE ANALYSIS
# ============================================================================

print("\n" + "="*60)
print("🌍 COUNTRY & ZONE ANALYSIS")
print("="*60)

if 'COUNTRY' in data.columns:
    print(f"\n📌 Unique Countries: {data['COUNTRY'].nunique()}")
    print("\n📈 TOP 15 COUNTRIES:")
    for country, cnt in data['COUNTRY'].value_counts().head(15).items():
        zone = COUNTRY_ZONES.get(country, 'Other')
        print(f"   {country}: {cnt:,} ({zone})")

if 'Zone' in data.columns:
    print(f"\n📌 ZONE DISTRIBUTION:")
    zone_dist = data['Zone'].value_counts()
    for zone, cnt in zone_dist.items():
        pct = 100 * cnt / len(data)
        print(f"   {zone}: {cnt:,} ({pct:.1f}%)")

    if 'FOB_VALUE' in data.columns:
        zone_fob = data.groupby('Zone')['FOB_VALUE'].sum().sort_values(ascending=False)
        total_fob = zone_fob.sum()
        print(f"\n📈 ZONE BY FOB VALUE:")
        for zone, val in zone_fob.head(10).items():
            pct = 100 * val / total_fob if total_fob > 0 else 0
            print(f"   {zone}: ₹{val:,.0f} ({pct:.1f}%)")

if 'Trade_Bloc' in data.columns:
    print(f"\n📌 TRADE BLOC DISTRIBUTION:")
    for bloc, cnt in data['Trade_Bloc'].value_counts().head(10).items():
        pct = 100 * cnt / len(data)
        print(f"   {bloc}: {cnt:,} ({pct:.1f}%)")

# ============================================================================
# CELL 17: PORT ANALYSIS
# ============================================================================

print("\n" + "="*60)
print("🚢 PORT ANALYSIS")
print("="*60)

if 'INDIAN_PORT' in data.columns:
    print(f"\n📌 Unique Indian Ports: {data['INDIAN_PORT'].nunique()}")
    print("\n📈 TOP 10 INDIAN PORTS:")
    for port, cnt in data['INDIAN_PORT'].value_counts().head(10).items():
        port_name = INDIAN_PORTS.get(str(port).upper(), str(port))
        print(f"   {port} ({port_name}): {cnt:,}")

if 'FOREIGN_PORT' in data.columns:
    print(f"\n📌 Unique Foreign Ports: {data['FOREIGN_PORT'].nunique()}")
    print("\n📈 TOP 10 FOREIGN PORTS:")
    for port, cnt in data['FOREIGN_PORT'].value_counts().head(10).items():
        print(f"   {port}: {cnt:,}")

# ============================================================================
# CELL 18: VALUE & VOLUME STATISTICS
# ============================================================================

print("\n" + "="*60)
print("💰 VALUE & VOLUME STATISTICS")
print("="*60)

if 'FOB_VALUE' in data.columns:
    print(f"\n📌 FOB VALUE STATISTICS:")
    print(f"   Total FOB: ₹{data['FOB_VALUE'].sum():,.0f}")
    print(f"   Average FOB: ₹{data['FOB_VALUE'].mean():,.0f}")
    print(f"   Median FOB: ₹{data['FOB_VALUE'].median():,.0f}")
    print(f"   Max FOB: ₹{data['FOB_VALUE'].max():,.0f}")

if 'QTY' in data.columns:
    print(f"\n📌 QUANTITY STATISTICS:")
    print(f"   Total Quantity: {data['QTY'].sum():,.0f}")
    print(f"   Average Quantity: {data['QTY'].mean():,.2f}")

if 'FOB_CATEGORY' in data.columns:
    print(f"\n📌 FOB CATEGORY BREAKDOWN:")
    for cat, cnt in data['FOB_CATEGORY'].value_counts().sort_index().items():
        pct = 100 * cnt / len(data)
        print(f"   {cat}: {cnt:,} ({pct:.1f}%)")

# ============================================================================
# CELL 19: DATA QUALITY SUMMARY
# ============================================================================

print("\n" + "="*60)
print("⚠️ DATA QUALITY FLAGS")
print("="*60)

quality_flags = {}
if 'IS_ZERO_FOB' in data.columns:
    quality_flags['Zero FOB'] = data['IS_ZERO_FOB'].sum()
if 'IS_SAMPLE' in data.columns:
    quality_flags['Samples (₹1-1K)'] = data['IS_SAMPLE'].sum()
if 'IS_LOW_VALUE' in data.columns:
    quality_flags['Low Value'] = data['IS_LOW_VALUE'].sum()
if 'IS_FOB_OUTLIER' in data.columns:
    quality_flags['FOB Outliers'] = data['IS_FOB_OUTLIER'].sum()
if 'IEC_STATUS' in data.columns:
    quality_flags['IEC Matched (not original)'] = (data['IEC_STATUS'] == 'MATCHED').sum()
    quality_flags['No IEC'] = (data['IEC_STATUS'] == 'NO_IEC').sum()

for flag, count in quality_flags.items():
    pct = 100 * count / len(data)
    print(f"   {flag}: {count:,} ({pct:.2f}%)")

# ============================================================================
# CELL 20: SAVE OUTPUTS
# ============================================================================

print("\n" + "="*60)
print("💾 SAVING OUTPUTS")
print("="*60)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'/content/Export_Analysis_Cleaned_{timestamp}.xlsx'

# Create summary DataFrames
summaries = {}

# Chapter Summary
if 'Chapter' in data.columns:
    agg_dict = {}
    if 'HS_CODE' in data.columns: agg_dict['HS_CODE'] = 'nunique'
    if 'EXPORTER' in data.columns: agg_dict['EXPORTER'] = 'nunique'
    if 'IMPORTER' in data.columns: agg_dict['IMPORTER'] = 'nunique'
    if 'COUNTRY' in data.columns: agg_dict['COUNTRY'] = 'nunique'
    if 'FOB_VALUE' in data.columns: agg_dict['FOB_VALUE'] = 'sum'

    if agg_dict:
        summaries['By_Chapter'] = data.groupby('Chapter').agg(agg_dict)
        summaries['By_Chapter']['Shipments'] = data.groupby('Chapter').size()
        summaries['By_Chapter'] = summaries['By_Chapter'].sort_values('Shipments', ascending=False)

# Zone Summary
if 'Zone' in data.columns:
    agg_dict = {}
    if 'COUNTRY' in data.columns: agg_dict['COUNTRY'] = 'nunique'
    if 'EXPORTER' in data.columns: agg_dict['EXPORTER'] = 'nunique'
    if 'FOB_VALUE' in data.columns: agg_dict['FOB_VALUE'] = 'sum'

    if agg_dict:
        summaries['By_Zone'] = data.groupby('Zone').agg(agg_dict)
        summaries['By_Zone']['Shipments'] = data.groupby('Zone').size()
        summaries['By_Zone'] = summaries['By_Zone'].sort_values('Shipments', ascending=False)

# Country Summary
if 'COUNTRY' in data.columns:
    agg_dict = {}
    if 'EXPORTER' in data.columns: agg_dict['EXPORTER'] = 'nunique'
    if 'IMPORTER' in data.columns: agg_dict['IMPORTER'] = 'nunique'
    if 'FOB_VALUE' in data.columns: agg_dict['FOB_VALUE'] = 'sum'

    if agg_dict:
        summaries['By_Country'] = data.groupby('COUNTRY').agg(agg_dict)
        summaries['By_Country']['Shipments'] = data.groupby('COUNTRY').size()
        summaries['By_Country']['Zone'] = summaries['By_Country'].index.map(COUNTRY_ZONES).fillna('Other')
        summaries['By_Country'] = summaries['By_Country'].sort_values('Shipments', ascending=False)

# Exporter Summary
if 'EXPORTER' in data.columns:
    agg_dict = {}
    if 'HS_CODE' in data.columns: agg_dict['HS_CODE'] = 'count'
    if 'FOB_VALUE' in data.columns: agg_dict['FOB_VALUE'] = 'sum'
    if 'COUNTRY' in data.columns: agg_dict['COUNTRY'] = 'nunique'

    if agg_dict:
        summaries['Top_Exporters'] = data.groupby('EXPORTER').agg(agg_dict)
        summaries['Top_Exporters'] = summaries['Top_Exporters'].sort_values(
            list(agg_dict.keys())[0], ascending=False
        ).head(200)

# HS Code Summary
if 'HS_CODE' in data.columns:
    agg_dict = {}
    if 'EXPORTER' in data.columns: agg_dict['EXPORTER'] = 'nunique'
    if 'IMPORTER' in data.columns: agg_dict['IMPORTER'] = 'nunique'
    if 'FOB_VALUE' in data.columns: agg_dict['FOB_VALUE'] = 'sum'

    if agg_dict:
        summaries['By_HS_Code'] = data.groupby('HS_CODE').agg(agg_dict)
        summaries['By_HS_Code']['Shipments'] = data.groupby('HS_CODE').size()
        summaries['By_HS_Code'] = summaries['By_HS_Code'].sort_values('Shipments', ascending=False)

# Save to Excel
with pd.ExcelWriter(output_file, engine='xlsxwriter') as writer:
    data.to_excel(writer, sheet_name='Cleaned_Data', index=False)
    for sheet_name, df in summaries.items():
        df.to_excel(writer, sheet_name=sheet_name)

print(f"✅ Saved: {output_file}")

# Download
files.download(output_file)

# ============================================================================
# CELL 21: FINAL SUMMARY
# ============================================================================

print("\n" + "="*60)
print("🎉 ANALYSIS COMPLETE!")
print("="*60)

total_fob = data['FOB_VALUE'].sum() if 'FOB_VALUE' in data.columns else 0

print(f"""
📊 FINAL STATISTICS
====================
Total Records: {len(data):,} (from {original_count:,} original)
Data Retention: {len(data)/original_count*100:.1f}%

HSN CODES:
  Chapters: {data['Chapter'].nunique() if 'Chapter' in data.columns else 'N/A'}
  Headings (HS4): {data['HS4'].nunique() if 'HS4' in data.columns else 'N/A'}
  Tariff Items (HS8): {data['HS_CODE'].nunique() if 'HS_CODE' in data.columns else 'N/A'}

PARTIES:
  Exporters: {data['EXPORTER'].nunique() if 'EXPORTER' in data.columns else 'N/A'}
  Importers: {data['IMPORTER'].nunique() if 'IMPORTER' in data.columns else 'N/A'}

GEOGRAPHY:
  Countries: {data['COUNTRY'].nunique() if 'COUNTRY' in data.columns else 'N/A'}
  Zones: {data['Zone'].nunique() if 'Zone' in data.columns else 'N/A'}

VALUE:
  Total FOB: ₹{total_fob:,.0f}

📁 Output Sheets:
  - Cleaned_Data (with all flags & features)
  - By_Chapter, By_Zone, By_Country
  - Top_Exporters, By_HS_Code
""")

print("\n✅ Data available as 'data' variable for further analysis")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 175.3/175.3 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 45.9 MB/s eta 0:00:00
✅ Dependencies installed!
📅 Session: 2026-02-05 04:07:58
📂 Upload EXPORT DATA FILE
   Required columns: HS CODE, PRODUCT DESCRIPTION, EXPORTER NAME,
   IMPORTER NAME, FOREIGN COUNTRY, FOB, QUANTITY, SB DATE, IEC, etc.


Saving combined files.xlsx to combined files.xlsx
✅ Data File: /content/combined files.xlsx

📂 Upload COUNTRIES REFERENCE FILE (Optional - for country standardization)
   Skip by pressing Cancel if not available


Saving countries.csv to countries.csv
✅ Countries File: /content/countries.csv
✅ Configuration & mappings loaded!
✅ Cleaning functions loaded!

📊 STAGE 1: LOADING & INITIAL CLEANING
✅ Loaded 89,765 records
📋 Columns: ['SB NO', 'HS4', 'CHAPTER', 'SB DATE', 'HS CODE', 'PRODUCT DESCRIPTION', 'QUANTITY', 'UNIT QUANTITY', 'UNIT RATE', 'CURRENCY', 'VALUE IN FC', 'FOB', 'FOREIGN PORT', 'FOREIGN COUNTRY', 'INDIAN PORT', 'IEC', 'EXPORTER NAME', 'Exporter Add1', 'Exporter Add2', 'Exporter City', 'IMPORTER NAME ']
✅ Replaced error values with NaN
✅ Column names standardized

📊 STAGE 2: DATA TYPE CONVERSIONS
✅ Date range: 2025-02-01 00:00:00 to 2025-08-30 00:00:00
✅ FOB range: ₹0 to ₹96,871,775
✅ Quantity range: 0 to 1,664,640
✅ HS Codes cleaned: 337 unique

📊 STAGE 3: REMOVING CRITICAL NULLS
✅ Removed 0 records with critical nulls (0.00%)
   Remaining: 89,765 records

📊 STAGE 4: IEC MATCHING
📊 IEC Statistics:
   Exporters with at least 1 IEC: 3,590
   Records with original IEC: 72,629
   Records 

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 ANALYSIS COMPLETE!

📊 FINAL STATISTICS
Total Records: 89,765 (from 89,765 original)
Data Retention: 100.0%

HSN CODES:
  Chapters: 1
  Headings (HS4): 12
  Tariff Items (HS8): 337

PARTIES:
  Exporters: 3956
  Importers: 5591

GEOGRAPHY:
  Countries: 179
  Zones: 13

VALUE:
  Total FOB: ₹155,458,633,236

📁 Output Sheets:
  - Cleaned_Data (with all flags & features)
  - By_Chapter, By_Zone, By_Country
  - Top_Exporters, By_HS_Code


✅ Data available as 'data' variable for further analysis


In [ ]:
"""
================================================================================
EXPORT DATA QUALITY CHECKER
================================================================================
Initial data profiling tool for export data files with varying column names.
- Handles multiple file formats (xlsx, xls, csv)
- Maps columns using semantic synonyms
- Generates quality reports before cleaning

For Google Colab - Upload folder or files, then run
================================================================================
"""

# ============================================================================
# CELL 1: INSTALL & IMPORT
# ============================================================================

!pip install -q pandas numpy openpyxl xlrd

import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

print("✅ Dependencies loaded!")
print(f"📅 Session: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")

# ============================================================================
# CELL 2: COLUMN SYNONYM MAPPING
# ============================================================================

# Canonical column names → list of possible variations found in different data sources
COLUMN_SYNONYMS = {
    # Product Information
    "PRODUCT_DESCRIPTION": [
        "PRODUCT DESCRIPTION", "ITEM", "ITEM DESCRIPTION",
        "ITEM DESCRIPTIONS", "PRODUCT_DESCRIPTION", "DESCRIPTION",
        "GOODS DESCRIPTION", "COMMODITY", "PRODUCT", "COMMODITY DESCRIPTION", "PRODUCTDESCRIPITION",
        "RITC DESCRIPTION", "PRODUCTDESCRIPITION","GOODSDESCRIPTION", "HSN_DESCRIPTION", 'ITEM_DESCRIPTION',
        'ITEM_CATEGORY_DESCRIPTION', "ITME", "PRODUCT_DESCRIPITION"
    ],
    "HS_CODE": [
        "HS CODE", "HS_CODE", "HSN_CODE", "RITC", "HSCODE",
        "HSN CODE", "HSN", "TARIFF CODE", "HS", "RITCCODE", "RITC_8", "RITC_CODE"
    ],
    "CHAPTER": [
        "CH", "CHAPTER", "2 DIGIT", "HSCODE(2 DIGIT)", "HS2",
        "CHAPTER CODE", "2DIGIT", "CHAPTER'S","CHEPTER"
    ],

    # Quantity & Value
    "QUANTITY": [
        "QUANTITY", "PRODUCT_QUANTITY", "QTY", "QTY."
    ],
    "UNIT": [
        "UQC", "UNIT", "QUANTITY_UNIT", "UNIT QUANTITY", "UOM",
        "UNIT OF MEASURE", "QUANTITY UNIT", "UNITOFMEASUREMENT", "UNITQUANTITY","UNIT_QUANTITY"
    ],
    "UNIT_RATE_FC": [
        "ITEM_RATE", "UNIT RATE IN FC", "UNT PRICE FC", "UNIT PRICE (USD)",
        "UNIT RATE", "UNIT PRICE", "RATE", "UNIT_VALUE_USD", "UNIT_VALUE_FC", "UNIT_RATE_USD",
        "UNITPRICE", "UNIT PRICE FOREIGN", "UNIT PRICE FC", "ITEM_RATE_IN_FC","UNIT RATE IN FOREIGN CURRENCY",

    ],
    "UNIT_RATE_INR": [
        "UNIT RATE IN INR", "UNIT PRICE IN INR", "RATE INR","PER UNIT FOB", "UNT PRICE INR",
        "ITEM_RATE", "UNIT_VALUE_INR", "UNIT_VALUE_IN_INR", "UNIT_RATE_IN_INR"
    ],
    "FOB_INR": [
        "FOB", "FOB INR", "FOB IN INR", "FOB VALUE (INR)", "FOB_IN_INR",
        "FOB VALUE", "FOB_VALUE", "VALUE INR", "INR VALUE", "FOB IN INR.1", "FOBVALUEINRS",
        "TOTAL_VALUE_IN_INR", "TOTAL FOB VALUE IN INR"
    ],
    "FOB_FC": [
        "FOB IN FC", "INV VALUE FC", "FOB FC", "VALUE IN FC",
        "FOB USD", "USD VALUE", "FC VALUE", "TOTAL VALUE IN FC",
        "TOTAL_VALUE_IN_FC", "TOTAL_VALUE_USD", "TOTAL_VALUE_FC","TOTAL_VALUE_IN_USD"
    ],
    "CURRENCY": [
        "CURRENCY", "CURR", "CUR", "CURRENCY CODE", "CURR", "CURRENCY_NAME","UNIT RATE CURRENCY"
    ],

    # Exporter Information
    "EXPORTER_NAME": [
        "EXPORTER", "EXPORTER NAME", "EXPORTER NAMES", "EXPORTER_NAME",
        "SHIPPER", "SHIPPER NAME", "SUPPLIER", "SELLER", "EXPORTER_PERSON_NAME", "EXPORTERNAME",
        "INDIAN EXPORTER NAME","EXPORTER_NAME"
    ],
    "EXPORTER_ID": [
        "EXPORTER ID", "IEC", "IEC CODE", "IEC NO", "EXPORTER_ID",
        "IE CODE", "IMPORTER EXPORTER CODE", "IECNO", "IEC_NO"
    ],
    "EXPORTER_ADDRESS": [
        "EXPORTER ADDRESS", "Exporter_Address", "EXPORTER ADD",
        "EXPORTER ADD1", "Exporter Add1", "SHIPPER ADDRESS", "SHIPPER'S ADDRESS", "EXPORTER_ADDRESS.1",
        "ADDRESS", "EXPORTER ADDRESS & STATE"
    ],
    "EXPORTER_CITY_STATE": [
        "EXPORTER CITY/ STATE", "EXPORTER CITY", "Exporter_City_State",
        "EXPORTER STATE", "Exporter City", "EXPORTER CITY/STATE", "CITY STATE", 'EXPORTER_CITY_STATE.1',
        'EXPORTER_STATE','CITY/ STATE',"EXPORTER_CITY_STATE","CITY/ STATE","EXPORTER_CITY_STATE",
        "EXPORTER ADD2"
    ],
    "EXPORTER_PINCODE": [
        "EXPORTER PIN", "Exporter_PIN", "EXPORTER PINCODE", "PIN CODE", "PIN", "EXPORTER_PIN",
        "PIN_CODE", 'EXPORTER PIN CODE','EXPORTER_PINCODE'
    ],
    "EXPORTER_CONTACT_PERSON": [
        "CONTACT PERSON", "CONTACT PERSON2", "Exporter_Person_Name", "CONTACT PERSON NAME",
    ],
    "EXPORTER_CONTACT_EMAIL": [
        "EMAIL ID", "Exporter_Email", "EMAIL", "E-MAIL", "EXPORTER EMAIL", "EXPORTER_EMAIL", "EMAILID",
        "EMAIL"
    ],
    "EXPORTER_CONTACT_PHONE": [
        "CONTACT NO.", "Exporter_Contact", "PHONE", "MOBILE", "CONTACT", "EXPORTER PHONE", "EXPORTER_PHONE",
        "PHONE", "EXPORTER_CONTACT", 'CONTACTNO'
    ],

    # Importer Information
    "IMPORTER_NAME": [
        "IMPORTER NAME", "IMPORTER NAMES", "IMPORTER_NAME", "CONSIGNEE",
        "IMPORTER", "BUYER", "BUYER NAME", "IMPORTER NAME ", "CONSIGNEE NAME",'CONSINEENAME','CONSIGNEENAME',
        "CONSINEE_NAME", "CONSIGNEE_NAME","FOREIGN IMPORTER NAME","CONSINEE_NAME","FOREIGN IMPORTER NAME ADDRESS"
    ],
    "IMPORTER_ADDRESS": [
        "IMPORTER ADDRESS", "Consignee_Address", "BUYER ADDRESS", "CONSIGNEE_ADDRESS",
        "CONSINEEADDRESS","CONSIGNEE_ADDRESS4", "ADDRESS2", "CONSIGNEE ADD","CONSIGNEEADDRESS",
        "CONSINEE_ADDRESS","FOR_ADD1",

    ],

    # Port & Location
    "INDIAN_PORT": [
        "PORT CODE", "ORIGIN PORT", "INDIAN PORT",
        "ORIGIN_PORT_CODE", "CUSH", "PORT", "LOADING PORT", "PORT OF LOADING",
        "LOCATION1", "LOCATION", "SOURCE_PORT", 'INDIAN_PORT', "INDIAN PORT NAME","PORT OF ORIGIN"
    ],
    "FOREIGN_PORT": [
        "FOREIGN PORT", "DESTINATION PORT", "DISCHARGE PORT",
        "POD", "PORT_CD", "PORT OF DISCHARGE",
        "DESTINATION_PORT", "FORIGN PORT",
        "FOREIGNPORT", "PORT OF DESTINATION","FOREIGN_PORT"
    ]
    ,
    "COUNTRY": [
        "COUNTRY", "FOREIGN COUNTRY", "DESTINATION_COUNTRY",
        "DEST COUNTRY", "DESTINATION", "COUNTRY OF DESTINATION",
        "COUNTRY OF ORIGIN", "CONSIGNEE COUNTRY", "SOURCE_COUNTRY", "ORIGIN_COUNTRY",
        "FOREIGNCOUNTRY", "COUNTRYOFDESTINATIONNAME", "FOREIGN_COUNTRY", "CTRY OF DESTINATION"
    ],
    "MODE_OF_PORT": [
        "MODE OF PORT", "SHIPMENT MODE", "MODE", "TRANSPORT MODE", "MODE_OF_TRANSPORT"
    ],

    # Date & Time
    "SB_DATE": [
        "SBDT", "SB DATE", "SBDATE", "SHIPPING_DATE", "SHIPPING BILL DATE",
        "SHIPPING DATE", "DATE", "EXPORT DATE", "SB_DATE", "DATE", "SB_DT"
    ],
    "MONTH": ["MONTH", "MON", "MM"],
    "YEAR": ["YEAR", "YR", "YYYY"],

    # Document References
    "SB_NO": [
        "SBNO", "SB NO", "SHIPPING BILL NO", "SHIPPING_BILL_NO",
        "SB NUMBER", "BILL NO", "SBNUMBER", "SYSTEM_ID", "SB.NO."
    ],
    "INVOICE_NO": [
        "INVOICE_NO", "INVOICE NO", "INV NO", "INVOICE NUMBER", "INVOICE NO.",'INVOICE_NUMBER'
    ],
    "ITEM_NO": ["ITEM_NO", "ITEM NO", "LINE NO", "SR NO", "ITEM NUMBER"],

    # Other
    "DRAWBACK": ["DRAWBACK", "DWARBACK", "DBK", "DRAWBAKDVALUE", "DWARBACK"],
    "TYPE": ["TYPE"],
    "UID": ["UID"],
    "ID": ["ID"],
}

print(f"✅ Column mappings loaded: {len(COLUMN_SYNONYMS)} canonical columns")

# ============================================================================
# CELL 3: FILE UPLOAD
# ============================================================================

from google.colab import files

print("="*60)
print("📂 Upload your export data files (xlsx, xls, csv)")
print("   You can upload multiple files at once")
print("="*60)

uploaded = files.upload()

# Save uploaded files to /content/data_files/
import os
UPLOAD_FOLDER = '/content/data_files'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)

for filename, content in uploaded.items():
    filepath = os.path.join(UPLOAD_FOLDER, filename)
    with open(filepath, 'wb') as f:
        f.write(content)
    print(f"✅ Saved: {filepath}")

print(f"\n📁 Total files uploaded: {len(uploaded)}")

# ============================================================================
# CELL 4: CORE FUNCTIONS
# ============================================================================

def read_all_files(folder_path):
    """Read all Excel/CSV files from a folder into a dictionary of DataFrames."""
    dataframes = {}

    for file in Path(folder_path).iterdir():
        try:
            if file.suffix.lower() in [".xlsx", ".xls"]:
                df = pd.read_excel(file)
            elif file.suffix.lower() == ".csv":
                # Try different encodings
                for encoding in ['utf-8', 'latin-1', 'cp1252']:
                    try:
                        df = pd.read_csv(file, encoding=encoding)
                        break
                    except:
                        continue
            else:
                continue

            # Standardize column names
            df.columns = (
                df.columns
                .astype(str)
                .str.strip()
                .str.upper()
            )

            dataframes[file.name] = df
            print(f"  ✅ {file.name}: {len(df):,} rows × {df.shape[1]} columns")

        except Exception as e:
            print(f"  ❌ {file.name}: Error - {str(e)[:50]}")

    return dataframes


def column_presence_matrix(dfs):
    """Create a matrix showing which columns exist in which files."""
    all_columns = sorted(
        set(col for df in dfs.values() for col in df.columns)
    )

    presence = pd.DataFrame(index=all_columns)

    for name, df in dfs.items():
        # Shorten filename for display
        short_name = name[:20] + "..." if len(name) > 20 else name
        presence[short_name] = presence.index.isin(df.columns)

    # Add count column
    presence['FILES_WITH_COLUMN'] = presence.sum(axis=1)
    presence = presence.sort_values('FILES_WITH_COLUMN', ascending=False)

    return presence


def semantic_column_coverage(dfs, synonym_map):
    """Check which canonical columns can be found in each file using synonyms."""
    records = []

    for file, df in dfs.items():
        cols = set(df.columns)

        for canonical, variants in synonym_map.items():
            found = cols.intersection(set(v.upper() for v in variants))
            records.append({
                "FILE": file,
                "CANONICAL_COLUMN": canonical,
                "FOUND": bool(found),
                "MATCHED_AS": ", ".join(found) if found else "-"
            })

    result = pd.DataFrame(records)
    return result


def semantic_coverage_summary(coverage_df):
    """Summarize which canonical columns are available across files."""
    summary = coverage_df.groupby('CANONICAL_COLUMN').agg({
        'FOUND': ['sum', 'count'],
        'MATCHED_AS': lambda x: ', '.join([m for m in x if m != '-'][:3])  # First 3 matches
    }).reset_index()

    summary.columns = ['CANONICAL_COLUMN', 'FILES_WITH_COLUMN', 'TOTAL_FILES', 'EXAMPLE_MATCHES']
    summary['COVERAGE_%'] = (summary['FILES_WITH_COLUMN'] / summary['TOTAL_FILES'] * 100).round(1)
    summary = summary.sort_values('COVERAGE_%', ascending=False)

    return summary


def basic_file_stats(df, filename):
    """Calculate basic statistics for a single DataFrame."""
    stats = {
        'FILE': filename,
        'ROWS': len(df),
        'COLUMNS': df.shape[1],
        'TOTAL_CELLS': df.size,
        'NULL_CELLS': df.isnull().sum().sum(),
        'NULL_%': round((df.isnull().sum().sum() / df.size) * 100, 2),
        'DUPLICATE_ROWS': df.duplicated().sum(),
        'DUPLICATE_%': round((df.duplicated().sum() / len(df)) * 100, 2) if len(df) > 0 else 0,
    }
    return stats


def value_field_analysis(df):
    """Analyze numeric/value fields for zeros, negatives, and outliers."""
    results = {}

    # Find value-related columns
    value_keywords = ["QUANTITY", "FOB", "RATE", "PRICE", "VALUE", "QTY", "AMOUNT"]
    value_cols = [
        c for c in df.columns
        if any(kw in c.upper() for kw in value_keywords)
    ]

    for col in value_cols:
        # Convert to numeric if not already
        if not pd.api.types.is_numeric_dtype(df[col]):
            numeric_col = pd.to_numeric(df[col].astype(str).str.replace(',', ''), errors='coerce')
        else:
            numeric_col = df[col]

        total = len(numeric_col)
        non_null = numeric_col.notna().sum()

        if non_null > 0:
            results[col] = {
                'TOTAL': total,
                'NON_NULL': non_null,
                'NULL': total - non_null,
                'ZEROS': (numeric_col == 0).sum(),
                'ZERO_%': round((numeric_col == 0).sum() / total * 100, 2),
                'NEGATIVES': (numeric_col < 0).sum(),
                'MIN': numeric_col.min(),
                'MAX': numeric_col.max(),
                'MEAN': round(numeric_col.mean(), 2),
                'MEDIAN': round(numeric_col.median(), 2),
            }

    return results


def date_field_analysis(df):
    """Analyze date fields for validity and range."""
    results = {}

    date_keywords = ["DATE", "DT", "SBDT"]
    date_cols = [c for c in df.columns if any(kw in c.upper() for kw in date_keywords)]

    for col in date_cols:
        parsed = pd.to_datetime(df[col], errors='coerce')
        valid = parsed.notna()

        results[col] = {
            'TOTAL': len(df),
            'VALID_DATES': valid.sum(),
            'INVALID_DATES': (~valid).sum(),
            'VALID_%': round(valid.sum() / len(df) * 100, 2),
            'MIN_DATE': parsed.min().strftime('%Y-%m-%d') if valid.any() else 'N/A',
            'MAX_DATE': parsed.max().strftime('%Y-%m-%d') if valid.any() else 'N/A',
        }

    return results


def text_field_analysis(df, synonym_map=None):
    """Analyze key text fields for uniqueness and patterns."""
    results = {}

    # Use full synonym map if provided, otherwise use defaults
    if synonym_map is None:
        synonym_map = COLUMN_SYNONYMS

    # Key fields to analyze
    key_fields = ['EXPORTER_NAME', 'IMPORTER_NAME', 'COUNTRY', 'HS_CODE', 'INDIAN_PORT', 'EXPORTER_ID']

    for field_name in key_fields:
        if field_name not in synonym_map:
            continue

        possible_cols = [v.upper() for v in synonym_map[field_name]]

        # Find ALL matching columns in this file
        matching_cols = [col for col in df.columns if col in possible_cols]

        if matching_cols:
            # Use first match for analysis, but report all matches
            col = matching_cols[0]
            series = df[col].astype(str).str.strip().str.upper()

            results[field_name] = {
                'COLUMN_FOUND': col,
                'ALL_MATCHES': matching_cols,  # Show all matching columns
                'TOTAL': len(series),
                'UNIQUE': series.nunique(),
                'NULL/EMPTY': series.isin(['', 'NAN', 'NONE', 'NA', '-']).sum(),
                'TOP_5': series.value_counts().head(5).to_dict()
            }

    return results


def generate_file_report(df, filename):
    """Generate comprehensive report for a single file."""
    return {
        'BASIC_STATS': basic_file_stats(df, filename),
        'VALUE_ANALYSIS': value_field_analysis(df),
        'DATE_ANALYSIS': date_field_analysis(df),
        'TEXT_ANALYSIS': text_field_analysis(df, COLUMN_SYNONYMS),
    }

print("✅ Analysis functions loaded!")

# ============================================================================
# CELL 5: RUN ANALYSIS
# ============================================================================

print("\n" + "="*60)
print("📊 RUNNING DATA QUALITY ANALYSIS")
print("="*60)

# Read all files
print("\n📂 Loading files...")
dfs = read_all_files(UPLOAD_FOLDER)

if not dfs:
    print("❌ No valid files found!")
else:
    print(f"\n✅ Loaded {len(dfs)} file(s)")

    # Column presence matrix
    print("\n" + "-"*60)
    print("📋 COLUMN PRESENCE ACROSS FILES")
    print("-"*60)
    col_matrix = column_presence_matrix(dfs)

    # Show columns present in ALL files vs some files
    all_files_cols = col_matrix[col_matrix['FILES_WITH_COLUMN'] == len(dfs)].index.tolist()
    some_files_cols = col_matrix[(col_matrix['FILES_WITH_COLUMN'] > 0) &
                                  (col_matrix['FILES_WITH_COLUMN'] < len(dfs))].index.tolist()

    print(f"\n✅ Columns in ALL files ({len(all_files_cols)}):")
    for col in all_files_cols[:20]:
        print(f"   • {col}")
    if len(all_files_cols) > 20:
        print(f"   ... and {len(all_files_cols) - 20} more")

    print(f"\n⚠️ Columns in SOME files ({len(some_files_cols)}):")
    for col in some_files_cols[:15]:
        count = col_matrix.loc[col, 'FILES_WITH_COLUMN']
        print(f"   • {col} ({count}/{len(dfs)} files)")
    if len(some_files_cols) > 15:
        print(f"   ... and {len(some_files_cols) - 15} more")

    # Semantic coverage
    print("\n" + "-"*60)
    print("🔗 SEMANTIC COLUMN MAPPING")
    print("-"*60)
    semantic_cov = semantic_column_coverage(dfs, COLUMN_SYNONYMS)
    semantic_summary = semantic_coverage_summary(semantic_cov)

    print("\n📌 Canonical Column Coverage:")
    print(semantic_summary[['CANONICAL_COLUMN', 'FILES_WITH_COLUMN', 'TOTAL_FILES', 'COVERAGE_%']].to_string(index=False))

    # Per-file reports
    print("\n" + "-"*60)
    print("📊 PER-FILE ANALYSIS")
    print("-"*60)

    file_reports = {}
    for filename, df in dfs.items():
        print(f"\n📄 {filename}")
        report = generate_file_report(df, filename)
        file_reports[filename] = report

        # Basic stats
        stats = report['BASIC_STATS']
        print(f"   Rows: {stats['ROWS']:,} | Columns: {stats['COLUMNS']}")
        print(f"   Nulls: {stats['NULL_%']}% | Duplicates: {stats['DUPLICATE_%']}%")

        # Value fields
        if report['VALUE_ANALYSIS']:
            print(f"\n   💰 Value Fields:")
            for col, vals in report['VALUE_ANALYSIS'].items():
                print(f"      {col}: Zeros={vals['ZERO_%']}%, Range=[{vals['MIN']:,.0f} - {vals['MAX']:,.0f}]")

        # Date fields
        if report['DATE_ANALYSIS']:
            print(f"\n   📅 Date Fields:")
            for col, vals in report['DATE_ANALYSIS'].items():
                print(f"      {col}: Valid={vals['VALID_%']}%, Range=[{vals['MIN_DATE']} to {vals['MAX_DATE']}]")

        # Text fields
        if report['TEXT_ANALYSIS']:
            print(f"\n   📝 Key Fields:")
            for field, vals in report['TEXT_ANALYSIS'].items():
                print(f"      {field}: {vals['UNIQUE']:,} unique values")


# ============================================================================
# CELL 6: SAVE REPORTS
# ============================================================================

print("\n" + "="*60)
print("💾 SAVING REPORTS")
print("="*60)

timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_file = f'/content/Data_Quality_Report_{timestamp}.xlsx'

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    # Column presence matrix
    col_matrix.to_excel(writer, sheet_name='Column_Presence')

    # Semantic coverage
    semantic_summary.to_excel(writer, sheet_name='Semantic_Coverage', index=False)
    semantic_cov.to_excel(writer, sheet_name='Semantic_Detail', index=False)

    # Basic stats summary
    basic_stats_list = [report['BASIC_STATS'] for report in file_reports.values()]
    pd.DataFrame(basic_stats_list).to_excel(writer, sheet_name='File_Summary', index=False)

    # Value analysis (combined)
    value_rows = []
    for filename, report in file_reports.items():
        for col, vals in report['VALUE_ANALYSIS'].items():
            row = {'FILE': filename, 'COLUMN': col, **vals}
            value_rows.append(row)
    if value_rows:
        pd.DataFrame(value_rows).to_excel(writer, sheet_name='Value_Analysis', index=False)

    # Date analysis (combined)
    date_rows = []
    for filename, report in file_reports.items():
        for col, vals in report['DATE_ANALYSIS'].items():
            row = {'FILE': filename, 'COLUMN': col, **vals}
            date_rows.append(row)
    if date_rows:
        pd.DataFrame(date_rows).to_excel(writer, sheet_name='Date_Analysis', index=False)

print(f"✅ Saved: {output_file}")

# Download
files.download(output_file)


# ============================================================================
# CELL 7: SUMMARY
# ============================================================================

print("\n" + "="*60)
print("🎉 DATA QUALITY CHECK COMPLETE!")
print("="*60)

print(f"""
📊 SUMMARY
==========
Files Analyzed: {len(dfs)}
Total Records: {sum(len(df) for df in dfs.values()):,}

Column Consistency:
  • Columns in ALL files: {len(all_files_cols)}
  • Columns in SOME files: {len(some_files_cols)}

Canonical Columns Found:
  • {len(semantic_summary[semantic_summary['COVERAGE_%'] == 100])} columns found in all files
  • {len(semantic_summary[semantic_summary['COVERAGE_%'] > 0])} columns found in at least one file

📁 Output: Data_Quality_Report_{timestamp}.xlsx
   • Column_Presence - Which columns exist in which files
   • Semantic_Coverage - Canonical column mapping
   • File_Summary - Basic stats per file
   • Value_Analysis - Numeric field quality
   • Date_Analysis - Date field quality

💡 NEXT STEPS:
1. Review Column_Presence to decide on common schema
2. Check Value_Analysis for zero/negative issues
3. Check Date_Analysis for parsing issues
4. Use semantic mappings to standardize column names
""")

✅ Dependencies loaded!
📅 Session: 2026-02-05 10:31:21
✅ Column mappings loaded: 34 canonical columns
📂 Upload your export data files (xlsx, xls, csv)
   You can upload multiple files at once


Saving 68 exp Feb2025.xlsx to 68 exp Feb2025.xlsx
Saving 68 exp JUL25.xlsx to 68 exp JUL25.xlsx
Saving 68 exp Mar 2025.xlsx to 68 exp Mar 2025.xlsx
Saving 68 Export June 25.xlsx to 68 Export June 25.xlsx
Saving EXP68MAY25.xlsx to EXP68MAY25.xlsx
✅ Saved: /content/data_files/68 exp Feb2025.xlsx
✅ Saved: /content/data_files/68 exp JUL25.xlsx
✅ Saved: /content/data_files/68 exp Mar 2025.xlsx
✅ Saved: /content/data_files/68 Export June 25.xlsx
✅ Saved: /content/data_files/EXP68MAY25.xlsx

📁 Total files uploaded: 5
✅ Analysis functions loaded!

📊 RUNNING DATA QUALITY ANALYSIS

📂 Loading files...
  ✅ 68 exp Feb2025.xlsx: 40,432 rows × 24 columns
  ✅ 68 Export June 25.xlsx: 44,274 rows × 28 columns
  ✅ 68 exp JUL25.xlsx: 51,203 rows × 27 columns
  ✅ 68 exp Mar 2025.xlsx: 50,869 rows × 25 columns
  ✅ EXP68MAY25.xlsx: 37,611 rows × 27 columns

✅ Loaded 5 file(s)

------------------------------------------------------------
📋 COLUMN PRESENCE ACROSS FILES
-----------------------------------------

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


🎉 DATA QUALITY CHECK COMPLETE!

📊 SUMMARY
Files Analyzed: 5
Total Records: 224,389

Column Consistency:
  • Columns in ALL files: 0
  • Columns in SOME files: 65

Canonical Columns Found:
  • 16 columns found in all files
  • 25 columns found in at least one file

📁 Output: Data_Quality_Report_20260205_103334.xlsx
   • Column_Presence - Which columns exist in which files
   • Semantic_Coverage - Canonical column mapping
   • File_Summary - Basic stats per file
   • Value_Analysis - Numeric field quality
   • Date_Analysis - Date field quality

💡 NEXT STEPS:
1. Review Column_Presence to decide on common schema
2. Check Value_Analysis for zero/negative issues
3. Check Date_Analysis for parsing issues
4. Use semantic mappings to standardize column names

